# VidFetch Telegram bot

**Original-quality video and audio, with large-file support. Everything runs in Colab.**

1. Use a standard **CPU** runtime (no GPU required).
2. Obtain your bot token from [@BotFather](https://t.me/BotFather), plus your own
   **api_id** and **api_hash** from [my.telegram.org/apps](https://my.telegram.org/apps).
   These are Telegram application credentials, separate from your bot token.
3. Stop any older copy of this bot. Select **Runtime → Run all** and enter the
   three credentials in the hidden prompts. The first setup builds Telegram's official
   server and can take 15 minutes or longer; rerunning setup reuses the completed build.
4. Open the bot link printed below, send `/start`, then a video URL.
5. Choose a quality: buttons include estimated video + audio size.
   Files are **not compressed to 50 MB**. Compatible videos play inline; other formats
   arrive as original-quality documents. **Audio only** preserves the original audio codec.

Telegram allows up to **2,000 MB per file** with this server. Files exceeding our
conservative 1,950 MB ceiling are sent as numbered parts plus a Python joiner.
Download all parts and the joiner into one folder and run the joiner with Python 3.
It verifies the original size and SHA-256; parts do not play separately.

Optionally add `TELEGRAM_BOT_TOKEN`, `TELEGRAM_API_ID`, and `TELEGRAM_API_HASH`
in Colab's **Secrets** panel and grant this notebook access to avoid retyping them.
Do not paste credentials into code cells. The API server runs only on Colab's loopback
interface; no public URL, tunnel, home computer server, or GPU is needed.

Keep the start cell running. Its stop button shuts the bot down; rerun it to restart.
Colab sessions are temporary and subject to [Colab's limits and policies](https://research.google.com/colaboratory/faq.html).
Interrupted downloads and old quality menus do not survive a runtime restart.
To update an older copy, stop its bot cell, reopen the latest GitHub notebook,
and run all cells again. Rerunning only the start cell does not update the embedded code.

Optional: import Netscape-format cookies in cell 2 using **Import / replace** for each site.
Export only `youtube.com` or `reddit.com` cookies, never paste their contents into a cell.
For Reddit, try a fresh signed-in Chrome export. For YouTube, follow the
[official export instructions](https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies).
Files are scoped to that site, checked for expiration and stored privately in this runtime.
`/cookies` shows status without cookie values. Stop the bot and rerun cell 2 to replace,
enable, disable or delete cookies; then restart cell 3 and send the link again.

Restrict access with Telegram user IDs in cell 3 (use `/id` to find yours).
**An explicit allowlist is required when account cookies are enabled.** Every allowed user
can request content accessible to the imported accounts; list only people you trust.
Leave the field blank to accept private messages from anyone who finds your bot.
Use `/cancel` to stop your request and `/status` to check it.

Sizes are estimates. Available disk space, network timeouts and runtime lifetime still apply.
Downloads and Telegram uploads use Colab's connection, not your home upload bandwidth.
Reddit uses Chrome impersonation and can read the requested post's own video player from an accessible public page if the JSON endpoint is rejected. Websites can still reject a
session or Colab's connection even with fresh cookies; cookies cannot guarantee access.
Download only content you are permitted to download.

[Source and documentation](https://github.com/HannanSeyfi/vidfetch-app/tree/main/bot)


In [ ]:
#@title 1. Install tools and load VidFetch
import importlib
import os
from pathlib import Path
import shutil
import subprocess
import sys
import sysconfig
import tempfile

if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--pre"]
               + ['httpx==0.28.1', 'yt-dlp[default,curl-cffi]>=2026.10.23.233055.dev0', 'deno>=2.3,<3', 'publicsuffixlist==1.1.2.20261023'], check=True)
os.environ["PATH"] = sysconfig.get_path("scripts") + os.pathsep + os.environ["PATH"]
encoders = subprocess.check_output(["ffmpeg", "-hide_banner", "-encoders"], text=True)
if "libx264" not in encoders:
    raise RuntimeError("This FFmpeg installation has no H.264 encoder (libx264).")
import deno
subprocess.run([deno.find_deno_bin(), "--version"], check=True)
# The notebook contains the same package as bot/vidfetch_bot in GitHub.
# Embedded sources make Open in Colab work from a branch, fork, or downloaded copy.
vidfetch_source_dir = Path(tempfile.mkdtemp(prefix="vidfetch-code-"))
for name, source in {'vidfetch_bot/__init__.py': '"""VidFetch\'s standalone Telegram companion. Android remains independent."""\n', 'vidfetch_bot/__main__.py': '"""Optional local entry point; Colab uses the same asynchronous run_bot function."""\n\nimport asyncio\nimport os\nfrom pathlib import Path\n\nfrom .app import run_bot\nfrom .cookies import CookieStore\nfrom .credentials import read_secret\nfrom .local_api import LocalAPIError, local_bot_api\nfrom .telegram import TelegramError\n\n\ndef main() -> None:\n    token = read_secret("TELEGRAM_BOT_TOKEN", "Telegram bot token from @BotFather: ")\n    try:\n        users = {\n            int(value.strip())\n            for value in os.environ.get("VIDFETCH_ALLOWED_USERS", "").split(",")\n            if value.strip()\n        }\n    except ValueError:\n        raise SystemExit(\n            "VIDFETCH_ALLOWED_USERS must contain comma-separated Telegram user IDs."\n        ) from None\n    cookie_directory = os.environ.get("VIDFETCH_COOKIE_DIR")\n    cookies = CookieStore(Path(cookie_directory)) if cookie_directory else None\n    if cookies and cookies.has_enabled() and not users:\n        raise SystemExit("Set VIDFETCH_ALLOWED_USERS before enabling account cookies.")\n    try:\n        api_id = read_secret("TELEGRAM_API_ID", "Telegram api_id from my.telegram.org: ")\n        api_hash = read_secret("TELEGRAM_API_HASH", "Telegram api_hash from my.telegram.org: ")\n\n        async def start():\n            binary = os.environ.get("VIDFETCH_API_BINARY")\n            async with local_bot_api(\n                token, api_id, api_hash, binary=Path(binary) if binary else None\n            ) as api:\n                await run_bot(token, allowed_users=users, api=api, cookie_store=cookies)\n\n        asyncio.run(start())\n    except KeyboardInterrupt:\n        pass\n    except (LocalAPIError, TelegramError) as error:\n        print(str(error))\n\n\nif __name__ == "__main__":\n    main()\n', 'vidfetch_bot/access.py': '"""Persistent, owner-managed access. An empty member list never makes a private bot public."""\n\nimport json\nimport os\nimport tempfile\nfrom pathlib import Path\n\nfrom .cookies import MAX_SITES, normalize_site\n\nMAX_USERS = 200\n\n\ndef valid_id(value: object) -> bool:\n    return type(value) is int and 0 < value < 2**63\n\n\nclass AccessStore:\n    def __init__(self, path: Path, owners: set[int], initial_users: set[int] | None = None):\n        if not owners or not all(valid_id(value) for value in owners):\n            raise ValueError("Configure at least one valid VIDFETCH_ADMIN_USERS owner ID.")\n        self.path = path\n        self.owners = frozenset(owners)\n        self.shares: dict[int, dict[str, int]] = {}\n        path.parent.mkdir(parents=True, exist_ok=True, mode=0o700)\n        path.parent.chmod(0o700)\n        if path.exists():\n            try:\n                if path.stat().st_size > 2_000_000:\n                    raise ValueError\n                payload = json.loads(path.read_text(encoding="utf-8"))\n                values = payload["allowed_users"]\n                if payload.get("version") not in {1, 2} or not isinstance(values, list):\n                    raise ValueError\n                if not all(valid_id(value) for value in values) or len(values) > MAX_USERS:\n                    raise ValueError\n                self.users = set(values) - self.owners\n                raw_shares = payload.get("cookie_shares", {})\n                if not isinstance(raw_shares, dict) or len(raw_shares) > MAX_USERS:\n                    raise ValueError\n                for recipient, grants in raw_shares.items():\n                    if not isinstance(recipient, str) or not recipient.isascii() or not recipient.isdecimal():\n                        raise ValueError\n                    recipient_text = recipient\n                    recipient = int(recipient)\n                    if str(recipient) != recipient_text:\n                        raise ValueError\n                    if not valid_id(recipient) or not self.allows(recipient) or not isinstance(grants, dict) or len(grants) > MAX_SITES + 1:\n                        raise ValueError\n                    for site, provider in grants.items():\n                        if site != "*" and normalize_site(site) != site:\n                            raise ValueError\n                        if not valid_id(provider) and not (type(provider) is int and provider == 0 and site != "*"):\n                            raise ValueError\n                    self.shares[recipient] = dict(grants)\n                self.path.chmod(0o600)\n            except (OSError, ValueError, KeyError, TypeError):\n                raise ValueError("The private access file is invalid. Restore it before starting.") from None\n        else:\n            users = set(initial_users or ()) - self.owners\n            self._save(users)\n\n    def allows(self, user_id: int) -> bool:\n        return user_id in self.owners or user_id in self.users\n\n    def _save(self, users: set[int], shares: dict[int, dict[str, int]] | None = None) -> None:\n        if not all(valid_id(value) for value in users) or len(users) > MAX_USERS:\n            raise ValueError(f"Use positive Telegram IDs; at most {MAX_USERS} users can be approved.")\n        shares = self.shares if shares is None else shares\n        shares = {user: dict(grants) for user, grants in shares.items() if user in users or user in self.owners}\n        if len(shares) > MAX_USERS or any(len(grants) > MAX_SITES + 1 for grants in shares.values()):\n            raise ValueError("Too many cookie-sharing rules. Remove an old rule first.")\n        payload = json.dumps({"version": 2, "allowed_users": sorted(users), "cookie_shares": {str(user): grants for user, grants in shares.items()}}) + "\\n"\n        if len(payload.encode()) > 2_000_000:\n            raise ValueError("The access configuration is too large.")\n        descriptor, name = tempfile.mkstemp(prefix="access-", dir=self.path.parent)\n        temporary = Path(name)\n        try:\n            with os.fdopen(descriptor, "w", encoding="utf-8") as stream:\n                stream.write(payload)\n                stream.flush()\n                os.fsync(stream.fileno())\n            temporary.replace(self.path)\n            self.users = set(users)\n            self.shares = shares\n        finally:\n            temporary.unlink(missing_ok=True)\n\n    def allow(self, users: set[int]) -> None:\n        self._save((self.users | users) - self.owners)\n\n    def deny(self, users: set[int]) -> None:\n        if users & self.owners:\n            raise ValueError("Owner access is configured on the server and cannot be removed here.")\n        self._save(self.users - users)\n\n    def share_cookies(self, recipient: int, provider: int, site: str = "*") -> None:\n        if not valid_id(recipient) or not self.allows(recipient):\n            raise ValueError("Approve this user with /allow ID first.")\n        if not valid_id(provider) or provider not in self.owners:\n            raise ValueError("Only a configured owner can share their cookie set.")\n        if recipient == provider:\n            raise ValueError("Your requests already use your own cookies.")\n        site = "*" if site in {"*", "all"} else normalize_site(site)\n        shares = {user: dict(grants) for user, grants in self.shares.items()}\n        grants = shares.setdefault(recipient, {})\n        if site == "*":\n            grants.clear()\n        grants[site] = provider\n        self._save(self.users, shares)\n\n    def unshare_cookies(self, recipient: int, site: str = "*") -> None:\n        site = "*" if site in {"*", "all"} else normalize_site(site)\n        shares = {user: dict(grants) for user, grants in self.shares.items()}\n        if site == "*":\n            shares.pop(recipient, None)\n        elif recipient in shares:\n            shares[recipient].pop(site, None)\n            if "*" in shares[recipient]:\n                shares[recipient][site] = 0  # An explicit exception to an entire-set grant.\n            if not shares[recipient]:\n                shares.pop(recipient)\n        self._save(self.users, shares)\n\n    def recipients_for_owner(self, provider: int) -> set[int]:\n        return {user for user, grants in self.shares.items() if provider in grants.values()}\n', 'vidfetch_bot/app.py': '"""Private-chat bot with expiring, owner-bound quality menus and bounded jobs."""\n\nfrom __future__ import annotations\n\nimport asyncio\nimport logging\nimport math\nimport re\nimport secrets\nimport shutil\nimport tempfile\nimport time\nfrom collections.abc import Callable\nfrom contextlib import aclosing\nfrom dataclasses import dataclass\nfrom html import escape\nfrom pathlib import Path\n\nfrom . import files, media\nfrom .access import AccessStore, valid_id\nfrom .cookie_routing import RoutedCookieStore\nfrom .cookies import CookieStore, UserCookieStores\nfrom .personal import OWNER_HELP, PersonalControls\nfrom .telegram import Telegram, TelegramError\n\nHELP = (\n    "Send a link and choose a video quality or Audio only. Each button shows an estimated file size; "\n    "video sizes include audio where available. Audio only extracts the original audio track. "\n    "Original quality is preserved with the local Telegram server. Files above its per-file limit "\n    "are delivered as numbered parts with a verified Python joiner. Download every part to rejoin.\\n\\n"\n    "Sizes are estimates. "\n    "Only one request is processed at a time by this bot.\\n\\n"\n    "/cancel — cancel your current request\\n/status — show your request status\\n"\n    "/id — show your Telegram user ID\\n/cookies — show site cookie status\\n/help — show these instructions"\n)\nPRIVATE_NOTICE = "This bot is private. Contact @sorrow_climax for approval. Send /id and include your Telegram user ID."\n\n\ndef delivery_caption(title: str, details: str, url: str) -> str:\n    """Keep the full source URL clickable, including URLs beyond the caption limit."""\n    visible = f"{title}\\n{details}\\n\\nSource: {url}"\n    # Telegram counts entities in UTF-16 units. Use a short link label when the\n    # full visible URL would exceed the caption budget; the href stays intact.\n    label = url if len(visible.encode("utf-16-le")) // 2 <= 1024 else "Original link"\n    return (\n        f"{escape(title)}\\n{escape(details)}\\n\\n"\n        f\'Source: <a href="{escape(url, quote=True)}">{escape(label)}</a>\'\n    )\n\n\n@dataclass\nclass Selection:\n    owner: tuple[int, int]\n    url: str\n    analysis: media.Analysis\n    expires: float\n\n\n@dataclass\nclass Job:\n    task: asyncio.Task\n    status: str = "Waiting in the queue…"\n\n\nclass Bot:\n    def __init__(\n        self,\n        api: Telegram,\n        *,\n        allowed_users: set[int] | None = None,\n        workspace: Path | None = None,\n        max_jobs: int = 5,\n        cookie_store: CookieStore | None = None,\n        access: AccessStore | None = None,\n        user_cookies: UserCookieStores | None = None,\n    ):\n        self.api = api\n        self.allowed_users = allowed_users or set()\n        if cookie_store and cookie_store.has_enabled() and not self.allowed_users:\n            raise ValueError("Set allowed_user_ids before enabling account cookies.")\n        self.cookie_store = cookie_store\n        if user_cookies and (not access or cookie_store):\n            raise ValueError("Personal cookies require private access and cannot share a global cookie store.")\n        self.access = access\n        self.workspace = workspace\n        self.max_jobs = max_jobs\n        self.jobs: dict[tuple[int, int], Job] = {}\n        self.selections: dict[str, Selection] = {}\n        self.slot = asyncio.Semaphore(1)\n        self.controls = PersonalControls(self, access, user_cookies) if user_cookies else None\n\n    def is_allowed(self, user_id: int) -> bool:\n        return self.access.allows(user_id) if self.access else not self.allowed_users or user_id in self.allowed_users\n\n    def cookie_options(self, owner: tuple[int, int]) -> dict:\n        if not self.is_allowed(owner[1]):\n            raise media.MediaError("Your access to this bot was removed.")\n        if self.controls:\n            return {"cookie_store": RoutedCookieStore(self.access, self.controls.stores, owner[1])}\n        if self.cookie_store and self.cookie_store.has_enabled() and not self.allowed_users:\n            raise media.MediaError(\n                "Account cookies require an explicit Telegram user allowlist. Stop the bot and set allowed_user_ids."\n            )\n        return {"cookie_store": self.cookie_store} if self.cookie_store else {}\n\n    def expire_menus(self, owner: tuple[int, int] | None = None) -> None:\n        now = time.monotonic()\n        self.selections = {\n            key: value\n            for key, value in self.selections.items()\n            if value.expires > now and value.owner != owner\n        }\n        # Bound memory even if many people discover an unrestricted bot.\n        while len(self.selections) >= 200:\n            self.selections.pop(next(iter(self.selections)))\n\n    async def handle(self, update: dict) -> None:\n        callback = update.get("callback_query")\n        message = callback.get("message", {}) if callback else update.get("message", {})\n        user = callback.get("from", {}) if callback else message.get("from", {})\n        chat = message.get("chat", {})\n        if not valid_id(user.get("id")) or not chat.get("id"):\n            return\n        if chat.get("type") != "private":\n            if callback:\n                await self.api.acknowledge(callback["id"], "Use the bot in a private chat.")\n            return\n        owner = (chat["id"], user["id"])\n        text = (message.get("text") or "").strip()\n        command = text.split(maxsplit=1)[0].split("@")[0].lower() if text else ""\n        if not callback and command == "/id":\n            await self.api.message(chat["id"], f"Your Telegram user ID: {user[\'id\']}")\n            return\n        if not self.is_allowed(user["id"]):\n            if callback:\n                await self.api.acknowledge(callback["id"], "This bot is private. Contact @sorrow_climax for approval." if self.access else "This bot is private.")\n            else:\n                await self.api.message(\n                    chat["id"], PRIVATE_NOTICE\n                    if self.access else "This bot is private."\n                )\n            return\n        if callback:\n            if self.controls and callback.get("data", "").startswith("c:"):\n                await self.controls.callback(callback, owner)\n            else:\n                await self.choose(callback, owner)\n            return\n        if self.controls and await self.controls.admin(owner, text or " "):\n            return\n        if self.controls and message.get("document"):\n            await self.controls.document(message, owner)\n            return\n        if self.controls and await self.controls.domain_message(owner, text):\n            return\n        if command in {"/start", "/help"}:\n            extra = OWNER_HELP if self.access and user["id"] in self.access.owners else ""\n            await self.api.message(chat["id"], HELP + extra)\n        elif command == "/cookies":\n            if self.controls:\n                argument = text.split(maxsplit=1)\n                if len(argument) == 2:\n                    await self.controls.start_import(owner, argument[1])\n                else:\n                    await self.controls.show(owner)\n                return\n            await self.api.message(\n                chat["id"],\n                self.cookie_store.status()\n                if self.cookie_store\n                else "No imported cookies. Configure optional cookies on the bot host.",\n            )\n        elif command == "/cancel":\n            if self.controls:\n                self.controls.pending.pop(owner, None)\n                self.controls.pending_domain.pop(owner, None)\n                self.controls.expire(owner)\n            self.expire_menus(owner)\n            job = self.jobs.get(owner)\n            if job:\n                job.task.cancel()\n                await asyncio.gather(job.task, return_exceptions=True)\n            await self.api.message(\n                chat["id"], "Cancelled." if job else "No active request. Old quality menus cleared."\n            )\n        elif command == "/status":\n            await self.api.message(\n                chat["id"],\n                self.jobs[owner].status\n                if owner in self.jobs\n                else "Ready. Send a link or choose a quality from your latest menu.",\n            )\n        elif text.startswith(("https://", "http://")) and len(text.split()) == 1:\n            await self.schedule(\n                owner, lambda status, folder: self.inspect(text, owner, status, folder)\n            )\n        else:\n            await self.api.message(chat["id"], "Send one video link, or use /help.")\n\n    async def schedule(\n        self, owner: tuple[int, int], action: Callable, *, status_message: dict | None = None\n    ) -> bool:\n        if not self.is_allowed(owner[1]):\n            await self.api.message(owner[0], PRIVATE_NOTICE if self.access else "This bot is private.")\n            return False\n        if owner in self.jobs:\n            await self.api.message(\n                owner[0], "You already have a request. Use /cancel before starting another."\n            )\n            return False\n        if len(self.jobs) >= self.max_jobs:\n            await self.api.message(\n                owner[0], "The queue is full. Please try again after a video finishes."\n            )\n            return False\n        notice = status_message or await self.api.message(owner[0], "Waiting in the queue…")\n\n        async def status(text: str) -> None:\n            if job := self.jobs.get(owner):\n                job.status = text\n            await self.api.edit(owner[0], notice["message_id"], text)\n\n        async def work() -> None:\n            try:\n                await status("Waiting in the queue…")\n                async with self.slot:\n                    if not self.is_allowed(owner[1]):\n                        await status("Your access to this bot was removed.")\n                        return\n                    with tempfile.TemporaryDirectory(\n                        prefix="vidfetch-", dir=self.workspace\n                    ) as folder:\n                        await action(status, Path(folder))\n            except asyncio.CancelledError:\n                await status("Cancelled.")\n                raise\n            except (media.MediaError, TelegramError) as error:\n                await status(str(error))\n            except Exception as error:\n                logging.getLogger(__name__).warning("Request failed (%s)", type(error).__name__)\n                await status("Something went wrong. Send the link again or try another quality.")\n            finally:\n                self.jobs.pop(owner, None)\n\n        task = asyncio.create_task(work())\n        self.jobs[owner] = Job(task)\n\n        def finished(done: asyncio.Task) -> None:\n            if (job := self.jobs.get(owner)) and job.task is done:\n                self.jobs.pop(owner, None)\n            if not done.cancelled() and (error := done.exception()):\n                logging.getLogger(__name__).warning("Request stopped (%s)", type(error).__name__)\n\n        task.add_done_callback(finished)\n        return True\n\n    async def inspect(\n        self, url: str, owner: tuple[int, int], status: media.Status, folder: Path\n    ) -> None:\n        await status("Finding available video qualities…")\n        options = self.cookie_options(owner)\n        analysis = await media.analyze(\n            url, folder, preserve_quality=self.api.local_files, **options\n        )\n        self.expire_menus(owner)\n        key = secrets.token_urlsafe(8)\n        self.selections[key] = Selection(owner, url, analysis, time.monotonic() + 1800)\n        keyboard = [\n            [\n                {\n                    "text": quality.label_for(\n                        self.api.upload_limit, preserve_quality=self.api.local_files\n                    ),\n                    "callback_data": f"q:{key}:{index}",\n                }\n            ]\n            for index, quality in enumerate(analysis.qualities)\n        ]\n        try:\n            await self.api.message(\n                owner[0],\n                f"{analysis.title}\\n\\nChoose a quality. Sizes include audio "\n                "when known. Audio only extracts the audio track. "\n                + (\n                    "Original quality is preserved. Files above the per-file limit are sent as "\n                    "numbered file parts that must be rejoined after download."\n                    if self.api.local_files\n                    else "Cloud mode compresses files to fit 49 MB."\n                )\n                + "\\n\\nMenu expires in 30 minutes.",\n                keyboard=keyboard,\n            )\n        except BaseException:\n            self.selections.pop(key, None)\n            raise\n        await status("Qualities ready. Choose a button in the message below.")\n\n    async def choose(self, callback: dict, owner: tuple[int, int]) -> None:\n        self.expire_menus()\n        match = re.fullmatch(r"q:([A-Za-z0-9_-]+):(\\d{1,3})", callback.get("data", ""))\n        selection = self.selections.get(match[1]) if match else None\n        index = int(match[2]) if match else -1\n        if (\n            not selection\n            or selection.owner != owner\n            or not 0 <= index < len(selection.analysis.qualities)\n        ):\n            await self.api.acknowledge(callback["id"], "This menu expired. Send the link again.")\n            return\n        await self.api.acknowledge(callback["id"])\n        quality = selection.analysis.qualities[index]\n\n        async def deliver(status: media.Status, folder: Path) -> None:\n            choice = (\n                "audio"\n                if quality.audio_only\n                else (f"{quality.height}p" if quality.height else "best available video")\n            )\n            await status(f"Downloading {choice}…")\n            options = self.cookie_options(owner)\n            source = await media.download(selection.url, quality, folder, **options)\n            if quality.audio_only:\n                audio = await media.prepare_audio(\n                    source,\n                    status,\n                    limit=self.api.upload_limit,\n                    preserve_quality=self.api.local_files,\n                )\n                size = audio.path.stat().st_size\n                if size > self.api.upload_limit:\n                    await self.deliver_parts(owner[0], audio.path, selection, status)\n                    return\n                await status(f"Uploading {media.format_size(size)} of audio to Telegram…")\n                details = f"Audio only · {media.format_size(size)}"\n                if audio.compressed:\n                    details += " · compressed for Telegram"\n                caption = delivery_caption(selection.analysis.title, details, selection.url)\n                if audio.document:\n                    await self.api.upload_document(owner[0], audio.path, caption)\n                else:\n                    await self.api.upload_audio(\n                        owner[0],\n                        audio.path,\n                        caption,\n                        title=selection.analysis.title,\n                        duration=math.ceil(audio.probe.duration),\n                    )\n                await status("Audio sent. Send another link whenever you\'re ready.")\n                return\n            prepared = await media.prepare_video(\n                source, status, limit=self.api.upload_limit, preserve_quality=self.api.local_files\n            )\n            size = prepared.path.stat().st_size\n            if size > self.api.upload_limit:\n                await self.deliver_parts(owner[0], prepared.path, selection, status)\n                return\n            await status(f"Uploading {media.format_size(size)} to Telegram…")\n            result_quality = f"{prepared.probe.height}p"\n            details = f"{result_quality} · {media.format_size(size)}"\n            if prepared.compressed:\n                details += " · compressed for Telegram"\n            caption = delivery_caption(selection.analysis.title, details, selection.url)\n            if prepared.document:\n                await self.api.upload_document(owner[0], prepared.path, caption)\n            else:\n                await self.api.upload(\n                    owner[0],\n                    prepared.path,\n                    caption,\n                    width=prepared.probe.width,\n                    height=prepared.probe.height,\n                    duration=math.ceil(prepared.probe.duration),\n                )\n            await status("Sent. Send another video link whenever you\'re ready.")\n\n        if await self.schedule(owner, deliver, status_message=callback["message"]):\n            self.selections.pop(match[1], None)\n\n    async def deliver_parts(\n        self, chat_id: int, path: Path, selection: Selection, status: media.Status\n    ) -> None:\n        if not self.api.local_files:\n            raise media.MediaError("The prepared file exceeds the cloud upload budget.")\n        name = f"VidFetch-{secrets.token_hex(4)}{path.suffix}"\n        size = path.stat().st_size\n        count = (size + self.api.upload_limit - 1) // self.api.upload_limit\n        await self.api.message(\n            chat_id,\n            f"This file needs {count} parts because Telegram limits individual files. "\n            "Quality is preserved. Download all parts and the joiner into one folder, then run "\n            "the joiner with Python 3 to restore the original file. Parts do not play separately.",\n        )\n        await status("Verifying the original file before sending its parts…")\n        digest = await files.file_hash(path)\n        async with aclosing(files.file_parts(path, self.api.upload_limit, name)) as parts:\n            async for part, index, total in parts:\n                await status(f"Uploading file part {index}/{total}…")\n                await self.api.upload_document(\n                    chat_id,\n                    part,\n                    delivery_caption(\n                        selection.analysis.title,\n                        f"Original quality · file part {index}/{total}",\n                        selection.url,\n                    ),\n                )\n        joiner = path.parent / f"join-{name}.py"\n        joiner.write_text(files.join_program(name, count, size, digest), encoding="utf-8")\n        await self.api.upload_document(\n            chat_id,\n            joiner,\n            "Save beside all parts and run with Python 3. "\n            "The original file is restored only after its size and SHA-256 match.",\n        )\n        await status(\n            f"Sent all {count} parts and the joiner. Download them together to restore the file."\n        )\n\n    async def cancel_user(self, user_id: int) -> None:\n        self.selections = {key: menu for key, menu in self.selections.items() if menu.owner[1] != user_id}\n        if self.controls:\n            self.controls.forget_controls(user_id)\n        tasks = [job.task for owner, job in self.jobs.items() if owner[1] == user_id]\n        for task in tasks:\n            task.cancel()\n        await asyncio.gather(*tasks, return_exceptions=True)\n\n    async def close(self) -> None:\n        tasks = [job.task for job in self.jobs.values()]\n        for task in tasks:\n            task.cancel()\n        await asyncio.gather(*tasks, return_exceptions=True)\n        self.jobs.clear()\n        self.selections.clear()\n        if self.controls:\n            self.controls.menus.clear()\n            self.controls.pending.clear()\n            self.controls.pending_domain.clear()\n\n\nasync def run_bot(\n    token: str,\n    *,\n    allowed_users: set[int] | None = None,\n    api: Telegram | None = None,\n    cookie_store: CookieStore | None = None,\n    access: AccessStore | None = None,\n    user_cookies: UserCookieStores | None = None,\n) -> None:\n    """Await directly in Colab; interrupt the running cell to shut down cleanly."""\n    if not re.fullmatch(r"\\d{5,20}:[A-Za-z0-9_-]{20,}", token.strip()):\n        print("Enter the complete bot token from @BotFather, including the colon.")\n        return\n    # Even DEBUG logging must not expose URLs containing Telegram credentials.\n    if cookie_store and cookie_store.has_enabled() and not allowed_users:\n        print(\n            "Set allowed_user_ids to your Telegram user ID before enabling account cookies. Use /id with cookies disabled to find it."\n        )\n        return\n    for name in ("httpx", "httpcore"):\n        logging.getLogger(name).setLevel(logging.CRITICAL)\n    for tool in ("ffmpeg", "ffprobe"):\n        if not shutil.which(tool):\n            print(f"Missing {tool}. Run the setup cell first.")\n            return\n    owns_api = api is None\n    api = api or Telegram(token.strip())\n    bot = Bot(api, allowed_users=allowed_users, cookie_store=cookie_store, access=access, user_cookies=user_cookies)\n    try:\n        identity = await api.call("getMe")\n        await api.call("deleteWebhook", {"drop_pending_updates": False})\n        await api.call(\n            "setMyCommands",\n            {\n                "commands": [\n                    {"command": "start", "description": "Send a link and choose video or audio"},\n                    {"command": "cancel", "description": "Cancel your current request"},\n                    {"command": "status", "description": "Show request status"},\n                    {"command": "id", "description": "Show your Telegram user ID"},\n                    {"command": "cookies", "description": "Manage your personal site cookies" if user_cookies else "Show site cookie status"},\n                    {"command": "help", "description": "Show instructions"},\n                ]\n            },\n        )\n        print(\n            f"VidFetch is running: https://t.me/{identity[\'username\']}\\n"\n            "Open that bot in Telegram and send /start, then a video link.\\n"\n            "Keep this process running. Stop it before starting another copy."\n        )\n        if access:\n            print(\n                f"Private access enabled: {len(access.owners)} owner(s), {len(access.users)} approved user(s). "\n                "Website cookie imports: /cookies DOMAIN. Owner cookie sharing: /sharecookies and /unsharecookies."\n            )\n        async for update in api.poll():\n            try:\n                await bot.handle(update)\n            except TelegramError as error:\n                if error.code in {401, 409}:\n                    raise\n    except TelegramError as error:\n        print(str(error))\n    except (asyncio.CancelledError, KeyboardInterrupt):\n        pass\n    finally:\n        await bot.close()\n        if owns_api:\n            await api.close()\n        print("VidFetch stopped. Start the process again to restart it.")\n', 'vidfetch_bot/colab_cookies.py': '"""Optional Colab uploads staged outside notebook source and outputs."""\n\nimport os\nimport tempfile\nfrom pathlib import Path\n\nfrom .cookies import CookieError, CookieStore\n\n\ndef configure_cookies(store: CookieStore, actions: dict[str, str]) -> None:\n    for site, action in actions.items():\n        try:\n            if action == "Import / replace":\n                from google.colab import files\n\n                print(f"Choose one Netscape cookies.txt export for {site}.")\n                previous = Path.cwd()\n                with tempfile.TemporaryDirectory(prefix="vidfetch-upload-") as directory:\n                    try:\n                        os.chdir(directory)\n                        uploaded = files.upload()\n                    finally:\n                        os.chdir(previous)\n                    if len(uploaded) != 1:\n                        raise CookieError("Choose exactly one cookie file for each site.")\n                    store.import_bytes(site, next(iter(uploaded.values())))\n                    del uploaded\n            elif action in {"Enable", "Disable"}:\n                store.set_enabled(site, action == "Enable")\n            elif action == "Delete":\n                store.delete(site)\n            elif action != "Keep":\n                raise CookieError("Choose a cookie action from the menu.")\n        except CookieError as error:\n            print(str(error))\n    print(store.status())\n    print("Cookie values are private. Enabled cookies require a user allowlist in the start cell.")\n', 'vidfetch_bot/cookie_routing.py': '"""Choose personal or explicitly delegated owner cookies for each request."""\n\nfrom contextlib import contextmanager\n\nfrom .access import AccessStore\nfrom .cookies import CookieError, UserCookieStores, site_for_url\n\n\nclass RoutedCookieStore:\n    def __init__(self, access: AccessStore, stores: UserCookieStores, user_id: int):\n        self.access = access\n        self.stores = stores\n        self.user_id = user_id\n\n    def _provider(self, url: str):\n        if not self.access.allows(self.user_id):\n            raise CookieError("Your access to this bot was removed.")\n        grants = self.access.shares.get(self.user_id, {})\n        explicit = site_for_url(url, [site for site in grants if site != "*"])\n        provider = grants.get(explicit) if explicit else grants.get("*")\n        if not provider:\n            return self.stores.for_user(self.user_id), False, None\n        if provider not in self.access.owners:\n            raise CookieError("The owner who shared these cookies is no longer available. Contact the bot owner.")\n        store = self.stores.for_user(provider)\n        site = site_for_url(url, store.sites())\n        if site is None and explicit:\n            raise CookieError(f"Owner-shared cookies for {explicit} are missing. Ask the owner to import them.")\n        return store, True, site\n\n    @contextmanager\n    def snapshot_for(self, url, folder):\n        store, shared, site = self._provider(url)\n        if not shared:\n            with store.snapshot_for(url, folder) as snapshot:\n                yield snapshot\n            return\n        if site is None:\n            # Whole-set mode uses guest access where the owner has no export, never another account.\n            yield None\n            return\n        if not store.is_enabled(site):\n            raise CookieError(f"Owner-shared cookies for {site} are disabled. Ask the owner to enable them.")\n        try:\n            with store.snapshot_site(site, folder) as snapshot:\n                yield snapshot\n        except CookieError:\n            raise CookieError(f"Owner-shared cookies for {site} are expired or invalid. Ask the owner to replace them.") from None\n\n    def shared_sites(self) -> tuple[str, ...]:\n        grants = self.access.shares.get(self.user_id, {})\n        sites = {site for site, provider in grants.items() if site != "*" and provider}\n        provider = grants.get("*")\n        if provider in self.access.owners:\n            sites.update(self.stores.for_user(provider).sites())\n        return tuple(sorted(sites))\n\n    def summary(self, sites) -> str:\n        grants = self.access.shares.get(self.user_id, {})\n        if not grants:\n            return "Cookie source: your own exports, or guest access when none are enabled."\n        lines = []\n        if "*" in grants:\n            lines.append("Owner-cookie mode: the owner\'s set is used, including future imports. Your exports are not used unless a site is unshared.")\n        for site in sites:\n            explicit = site_for_url("https://" + site, [key for key in grants if key != "*"])\n            provider = grants.get(explicit) if explicit else grants.get("*")\n            lines.append(f"{site}: " + ("owner-shared cookies" if provider else "your own cookies / guest"))\n        return "\\n".join(lines)\n', 'vidfetch_bot/cookies.py': '"""Private, domain-scoped cookie imports. Never include cookie values in messages."""\n\nfrom __future__ import annotations\n\nimport contextlib\nimport hashlib\nimport ipaddress\nimport os\nimport re\nimport tempfile\nimport time\nfrom pathlib import Path\nfrom urllib.parse import urlsplit\n\nfrom publicsuffixlist import PublicSuffixList\n\nMAX_COOKIE_BYTES = 2_000_000\nSITES = ("youtube.com", "reddit.com")\nMAX_SITES = 25\n_PSL = PublicSuffixList()\n\n\nclass CookieError(ValueError):\n    pass\n\n\ndef domain_matches(host: str, domain: str) -> bool:\n    return host == domain or host.endswith("." + domain)\n\n\ndef normalize_site(value: str) -> str:\n    """A single website, never a public suffix, IP, URL credential or filesystem path."""\n    if not isinstance(value, str) or not value.strip() or len(value) > 2048:\n        raise CookieError("Enter a website domain, such as youtube.com or example.com.")\n    value = value.strip().removeprefix(".")\n    value = {"youtube": "youtube.com", "reddit": "reddit.com"}.get(value.lower(), value)\n    try:\n        parsed = urlsplit(value if "://" in value else "https://" + value)\n        if parsed.scheme not in {"http", "https"} or parsed.username or parsed.password or parsed.port:\n            raise ValueError\n        if parsed.path not in {"", "/"} or parsed.query or parsed.fragment:\n            raise ValueError\n        host = (parsed.hostname or "").encode("idna").decode("ascii").lower().rstrip(".")\n        if len(host) > 253 or not re.fullmatch(r"[a-z0-9](?:[a-z0-9-]*[a-z0-9])?(?:\\.[a-z0-9](?:[a-z0-9-]*[a-z0-9])?)+", host):\n            raise ValueError\n        if any(len(label) > 63 for label in host.split(".")) or not _PSL.privatesuffix(host):\n            raise ValueError\n        if not re.search(r"[a-z]", host.split(".")[-1]):\n            raise ValueError\n        try:\n            ipaddress.ip_address(host)\n        except ValueError:\n            pass\n        else:\n            raise ValueError\n        if host.endswith((".localhost", ".local", ".internal", ".invalid", ".test", ".onion")):\n            raise ValueError\n        if host.startswith("www.") and host[4:] == _PSL.privatesuffix(host):\n            host = host[4:]\n    except (ValueError, UnicodeError):\n        raise CookieError("Use a website domain, not an IP, public suffix, path or address with a port.") from None\n    return canonical_host(host)\n\n\ndef canonical_host(host: str) -> str:\n    if host == "youtu.be" or any(domain_matches(host, item) for item in ("youtube.com", "youtube-nocookie.com")):\n        return "youtube.com"\n    if any(domain_matches(host, item) for item in ("reddit.com", "redditmedia.com", "redd.it")):\n        return "reddit.com"\n    return host\n\n\ndef site_for_url(value: str, sites=None) -> str | None:\n    try:\n        parsed = urlsplit(value if "://" in value else "https://" + value)\n        if parsed.scheme not in {"https", "http"} or parsed.username or parsed.password:\n            return None\n        host = (parsed.hostname or "").encode("idna").decode().lower().rstrip(".")\n    except (ValueError, UnicodeError):\n        return None\n    host = canonical_host(host)\n    matches = [site for site in (SITES if sites is None else sites) if domain_matches(host, site)]\n    return max(matches, key=len) if matches else None\n\n\ndef scoped_cookies(data: bytes, site: str) -> str:\n    site = normalize_site(site)\n    if len(data) > MAX_COOKIE_BYTES:\n        raise CookieError("Cookie files must be smaller than 2 MB.")\n    try:\n        text = data.decode("utf-8-sig")\n    except UnicodeError:\n        raise CookieError("Export a UTF-8 Netscape cookies.txt file.") from None\n    if not text.startswith(("# Netscape HTTP Cookie File", "# HTTP Cookie File")):\n        raise CookieError("Export cookies in Netscape cookies.txt format.")\n    rows = []\n    now = time.time()\n    for line in text.splitlines():\n        if not line.strip() or (line.startswith("#") and not line.startswith("#HttpOnly_")):\n            continue\n        fields = line.removeprefix("#HttpOnly_").split("\\t")\n        if (\n            len(fields) != 7\n            or fields[1] not in {"TRUE", "FALSE"}\n            or fields[3] not in {"TRUE", "FALSE"}\n            or not fields[2].startswith("/")\n            or not fields[4].isascii()\n            or not fields[4].isdigit()\n            or len(fields[4]) > 20\n            or (fields[0].startswith(".") != (fields[1] == "TRUE"))\n            or not re.fullmatch(r"[A-Za-z0-9_-]+(?:\\.[A-Za-z0-9_-]+)*", fields[0].lstrip("."))\n            or not fields[5]\n            or "\\x00" in line\n        ):\n            raise CookieError("The file contains an invalid Netscape cookie entry.")\n        domain = fields[0].lstrip(".").lower().rstrip(".")\n        expires = int(fields[4])\n        if domain_matches(domain, site) and (expires == 0 or expires > now):\n            rows.append(line)\n    if not rows:\n        raise CookieError(f"No unexpired cookies for {site}. Check the domain or export a fresh signed-in session.")\n    result = "# Netscape HTTP Cookie File\\n" + "\\n".join(rows) + "\\n"\n    return result\n\n\nclass CookieStore:\n    def __init__(self, directory: Path):\n        self.directory = directory\n        directory.mkdir(mode=0o700, parents=True, exist_ok=True)\n        directory.chmod(0o700)\n\n    def _path(self, site: str) -> Path:\n        site = normalize_site(site)\n        stem = site if len(site) <= 240 else "site-" + hashlib.sha256(site.encode()).hexdigest()\n        return self.directory / f"{stem}.cookies"\n\n    def sites(self) -> tuple[str, ...]:\n        found = set()\n        for path in self.directory.glob("*.cookies"):\n            if path.is_symlink() or not path.is_file():\n                continue\n            try:\n                name = path.stem\n                if name.startswith("site-") and "." not in name:\n                    with path.open("rb") as stream:\n                        header = stream.read(1024)\n                    name = next(line.removeprefix(b"# VidFetch site: ").decode("ascii") for line in header.splitlines() if line.startswith(b"# VidFetch site: "))\n                site = normalize_site(name)\n                if self._path(site) == path:\n                    found.add(site)\n            except (CookieError, OSError, UnicodeError, StopIteration):\n                continue\n        return tuple(sorted(found))\n\n    def import_file(self, site: str, path: Path) -> None:\n        with path.open("rb") as source:\n            data = source.read(MAX_COOKIE_BYTES + 1)\n        self.import_bytes(site, data)\n\n    def import_bytes(self, site: str, data: bytes) -> None:\n        site = normalize_site(site)\n        if not self.has_site(site) and len(self.sites()) >= MAX_SITES:\n            raise CookieError(f"You can save cookies for {MAX_SITES} sites. Remove an old site first.")\n        value = scoped_cookies(data, site)\n        if len(site) > 240:\n            value = value.replace("# Netscape HTTP Cookie File\\n", f"# Netscape HTTP Cookie File\\n# VidFetch site: {site}\\n", 1)\n        descriptor, name = tempfile.mkstemp(prefix="import-", dir=self.directory)\n        temporary = Path(name)\n        try:\n            with os.fdopen(descriptor, "w", encoding="utf-8", newline="\\n") as stream:\n                stream.write(value)\n            temporary.replace(self._path(site))\n            self._marker(site).unlink(missing_ok=True)\n        finally:\n            temporary.unlink(missing_ok=True)\n\n    def set_enabled(self, site: str, enabled: bool) -> None:\n        site = normalize_site(site)\n        if not self._path(site).is_file():\n            raise CookieError(f"Import {site} cookies first.")\n        marker = self._marker(site)\n        if enabled:\n            marker.unlink(missing_ok=True)\n        else:\n            marker.touch(mode=0o600)\n\n    def delete(self, site: str) -> None:\n        self._path(site).unlink(missing_ok=True)\n        self._marker(site).unlink(missing_ok=True)\n\n    def _marker(self, site: str) -> Path:\n        return self._path(site).with_suffix(".disabled")\n\n    def has_enabled(self) -> bool:\n        return any(self._enabled(site) for site in self.sites())\n\n    def has_site(self, site: str) -> bool:\n        return self._path(site).is_file()\n\n    def is_enabled(self, site: str) -> bool:\n        return self._enabled(site)\n\n    def _enabled(self, site: str) -> bool:\n        return self._path(site).is_file() and not self._marker(site).exists()\n\n    def status(self, sites=None) -> str:\n        lines = []\n        for site in (tuple(dict.fromkeys((*SITES, *self.sites()))) if sites is None else sites):\n            if not self._path(site).exists():\n                lines.append(f"{site}: no imported cookies")\n                continue\n            state = "enabled" if self._enabled(site) else "disabled"\n            try:\n                value = scoped_cookies(self._path(site).read_bytes(), site)\n                rows = value.splitlines()[1:]\n                names = {line.removeprefix("#HttpOnly_").split("\\t")[5] for line in rows}\n                account = {"reddit.com": {"reddit_session"}, "youtube.com": {"SID", "__Secure-1PSID", "__Secure-3PSID"}}.get(site)\n                session = ("sign-in cookie present" if names & account else "no sign-in cookie found") if account else "session validity unverified"\n                lines.append(f"{site}: {state}, {len(rows)} unexpired cookies, {session}")\n            except CookieError:\n                lines.append(f"{site}: {state}, expired or invalid; import a fresh export")\n        return "\\n".join(lines)\n\n    @contextlib.contextmanager\n    def snapshot_for(self, url: str, folder: Path):\n        site = site_for_url(url, self.sites())\n        if site is None or not self._enabled(site):\n            yield None\n            return\n        with self.snapshot_site(site, folder) as snapshot:\n            yield snapshot\n\n    @contextlib.contextmanager\n    def snapshot_site(self, site: str, folder: Path):\n        value = scoped_cookies(self._path(site).read_bytes(), site)\n        descriptor, name = tempfile.mkstemp(prefix="session-", suffix=".cookies", dir=folder)\n        snapshot = Path(name)\n        try:\n            with os.fdopen(descriptor, "w", encoding="utf-8", newline="\\n") as stream:\n                stream.write(value)\n            yield snapshot\n        finally:\n            snapshot.unlink(missing_ok=True)\n\n\nclass UserCookieStores:\n    """Numeric user IDs select private stores; uploaded filenames never select paths."""\n\n    def __init__(self, root: Path):\n        self.root = root.resolve()\n        self.root.mkdir(parents=True, exist_ok=True, mode=0o700)\n        self.root.chmod(0o700)\n\n    def for_user(self, user_id: int) -> CookieStore:\n        if type(user_id) is not int or not 0 < user_id < 2**63:\n            raise CookieError("A valid Telegram user ID is required.")\n        folder = self.root / str(user_id)\n        if folder.is_symlink() or folder.resolve().parent != self.root:\n            raise CookieError("The personal cookie directory is unavailable.")\n        return CookieStore(folder)\n\n    def forget(self, user_id: int) -> None:\n        store = self.for_user(user_id)\n        for site in set(SITES) | set(store.sites()):\n            store.delete(site)\n        try:\n            store.directory.rmdir()\n        except OSError:\n            pass  # Only the two known exports/markers are removed; never recurse.\n', 'vidfetch_bot/credentials.py': '"""Read credentials without putting them in notebook source or output."""\n\nimport os\nfrom getpass import getpass\n\n\ndef read_secret(name: str, prompt: str) -> str:\n    value = os.environ.get(name, "").strip()\n    if not value:\n        try:\n            from google.colab import userdata\n\n            value = str(userdata.get(name) or "").strip()\n        except Exception:\n            # Missing Colab secret, denied permission, or a non-Colab environment.\n            # Do not print exception text, which is controlled by the secret provider.\n            value = ""\n    return value or getpass(prompt).strip()\n', 'vidfetch_bot/files.py': '"""Bounded-memory, byte-exact delivery of files above Telegram\'s per-message ceiling."""\n\nfrom __future__ import annotations\n\nimport asyncio\nimport hashlib\nfrom pathlib import Path\n\nfrom .media import check_disk\n\nBLOCK = 8 * 1024 * 1024\n\n\nasync def file_hash(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        while block := stream.read(BLOCK):\n            digest.update(block)\n            await asyncio.sleep(0)\n    return digest.hexdigest()\n\n\nasync def file_parts(source: Path, limit: int, name: str):\n    if limit <= 0 or Path(name).name != name:\n        raise ValueError("Use a positive part size and a plain filename.")\n    count = (source.stat().st_size + limit - 1) // limit\n    with source.open("rb") as stream:\n        for index in range(1, count + 1):\n            path = source.parent / f"{name}.part{index:04d}"\n            created = False\n            try:\n                remaining = limit\n                with path.open("xb") as part:\n                    created = True\n                    while remaining and (block := stream.read(min(remaining, BLOCK))):\n                        check_disk(source.parent)\n                        part.write(block)\n                        remaining -= len(block)\n                        await asyncio.sleep(0)\n                yield path, index, count\n            finally:\n                if created:\n                    path.unlink(missing_ok=True)\n\n\ndef join_program(name: str, count: int, size: int, digest: str) -> str:\n    """A dependency-free joiner verifies every byte before creating the completed media file."""\n    return f\'\'\'"""Save this script beside all {count} parts and run it with Python 3."""\nimport hashlib\nfrom pathlib import Path\n\nfolder = Path(__file__).resolve().parent\nname = {name!r}\nexpected_size = {size}\nexpected_hash = {digest!r}\nparts = [folder / (name + ".part%04d" % n) for n in range(1, {count + 1})]\nif any(not part.is_file() for part in parts):\n    raise SystemExit("Missing parts. Download every part into the same folder as this script.")\noutput = folder / name\nif output.exists():\n    raise SystemExit("The output already exists. Move it before joining again.")\ntemporary = folder / (name + ".joining")\ndigest = hashlib.sha256()\ncreated = False\ntry:\n    with temporary.open("xb") as result:\n        created = True\n        for part in parts:\n            with part.open("rb") as source:\n                while block := source.read(8 * 1024 * 1024):\n                    result.write(block)\n                    digest.update(block)\n    if temporary.stat().st_size != expected_size or digest.hexdigest() != expected_hash:\n        raise SystemExit("Parts are incomplete or corrupt. Download them again.")\n    temporary.rename(output)\n    print("Restored original file:", output.name)\nfinally:\n    if created:\n        temporary.unlink(missing_ok=True)\n\'\'\'\n', 'vidfetch_bot/local_api.py': '"""Install and supervise Telegram\'s official Bot API server on the bot\'s own machine."""\n\nfrom __future__ import annotations\n\nimport asyncio\nimport logging\nimport os\nimport re\nimport shutil\nimport socket\nimport subprocess\nimport sys\nimport tempfile\nimport time\nfrom contextlib import asynccontextmanager\nfrom pathlib import Path\n\nfrom .telegram import Telegram, TelegramError\n\nAPI_REVISION = "e3e9dd8e5b3d7ab8537cd5a10dc31d5ffa8f82d1"\n\n\nclass LocalAPIError(Exception):\n    """A credential-free error suitable for notebook output."""\n\n\ndef install_server(cache: Path | None = None) -> Path:\n    """Build pinned official sources once per Colab runtime; never download third-party binaries."""\n    root = (cache or Path.home() / ".cache" / "vidfetch-bot-api") / API_REVISION\n    binary = root / "install" / "bin" / "telegram-bot-api"\n    if binary.is_file():\n        return binary\n    if not sys.platform.startswith("linux"):\n        raise LocalAPIError("Automatic server installation requires Linux (including Colab).")\n    root.mkdir(parents=True, exist_ok=True)\n    source = root / "source"\n    apt = ([] if getattr(os, "geteuid", lambda: 0)() == 0 else ["sudo"]) + ["apt-get"]\n    steps = [\n        apt + ["update", "-qq"],\n        [\n            *apt,\n            "install",\n            "-y",\n            "-qq",\n            "git",\n            "cmake",\n            "clang",\n            "make",\n            "gperf",\n            "libssl-dev",\n            "zlib1g-dev",\n        ],\n    ]\n    if not (source / ".git").is_dir():\n        steps.append(\n            [\n                "git",\n                "clone",\n                "--no-checkout",\n                "https://github.com/tdlib/telegram-bot-api.git",\n                str(source),\n            ]\n        )\n    steps.extend(\n        [\n            ["git", "-C", str(source), "checkout", "--detach", API_REVISION],\n            [\n                "git",\n                "-C",\n                str(source),\n                "submodule",\n                "update",\n                "--init",\n                "--recursive",\n                "--depth",\n                "1",\n            ],\n            [\n                "cmake",\n                "-S",\n                str(source),\n                "-B",\n                str(root / "build"),\n                "-DCMAKE_BUILD_TYPE=Release",\n                "-DCMAKE_C_COMPILER=clang",\n                "-DCMAKE_CXX_COMPILER=clang++",\n                f"-DCMAKE_INSTALL_PREFIX={root / \'install\'}",\n            ],\n            ["cmake", "--build", str(root / "build"), "--target", "install", "--parallel", "2"],\n        ]\n    )\n    print(\n        "Building Telegram\'s official API server. First setup can take 15 minutes or longer; "\n        "completed builds are reused within this runtime."\n    )\n    for args in steps:\n        subprocess.run(args, check=True)\n    if not binary.is_file():\n        raise LocalAPIError("The Telegram API server build did not produce an executable.")\n    return binary\n\n\nasync def wait_for_server(process, port: int, *, timeout: float = 90) -> None:\n    deadline = time.monotonic() + timeout\n    while time.monotonic() < deadline:\n        if process.returncode is not None:\n            raise LocalAPIError("The local Telegram API server stopped during startup.")\n        try:\n            _, writer = await asyncio.open_connection("127.0.0.1", port)\n            writer.close()\n            await writer.wait_closed()\n            return\n        except OSError:\n            await asyncio.sleep(0.25)\n    raise LocalAPIError("The local Telegram API server did not become ready. Restart the bot cell.")\n\n\nasync def stop_server(process) -> None:\n    if process.returncode is None:\n        process.terminate()\n        try:\n            await asyncio.wait_for(process.wait(), 20)\n        except TimeoutError:\n            process.kill()\n            await process.wait()\n\n\nasync def migrate_from_cloud(token: str) -> None:\n    for name in ("httpx", "httpcore"):\n        logging.getLogger(name).setLevel(logging.CRITICAL)\n    cloud = Telegram(token)\n    try:\n        await cloud.call("logOut")\n    except TelegramError as error:\n        # A fresh Colab runtime may start during the cloud\'s ten-minute login cooldown.\n        # Local login is allowed immediately after logout; do not fall back to cloud uploads.\n        if error.code != 429 or not 0 < error.retry_after <= 600:\n            raise\n    finally:\n        await cloud.close()\n\n\n@asynccontextmanager\nasync def local_bot_api(\n    token: str,\n    api_id: str,\n    api_hash: str,\n    *,\n    binary: Path | None = None,\n    directory: Path | None = None,\n):\n    if not re.fullmatch(r"\\d{5,20}:[A-Za-z0-9_-]{20,}", token):\n        raise LocalAPIError("Enter the complete bot token from @BotFather.")\n    if not api_id.isdecimal() or int(api_id) <= 0 or not re.fullmatch(r"[a-fA-F0-9]{32}", api_hash):\n        raise LocalAPIError(\n            "Enter the numeric api_id and 32-character api_hash from my.telegram.org."\n        )\n    executable = str(binary or shutil.which("telegram-bot-api") or "")\n    if not executable or not Path(executable).is_file():\n        raise LocalAPIError("The Telegram API server is missing. Run the setup cell first.")\n    # This directory survives stopping/rerunning a cell, but not deleting the Colab runtime.\n    root = directory or Path(tempfile.gettempdir()) / "vidfetch-telegram-api" / token.split(":")[0]\n    root.mkdir(parents=True, exist_ok=True, mode=0o700)\n    os.chmod(root, 0o700)\n    with socket.socket() as probe:\n        probe.bind(("127.0.0.1", 0))\n        port = probe.getsockname()[1]\n    environment = {**os.environ, "TELEGRAM_API_ID": api_id, "TELEGRAM_API_HASH": api_hash}\n    process = None\n    api = None\n    try:\n        process = await asyncio.create_subprocess_exec(\n            executable,\n            "--local",\n            "--http-ip-address=127.0.0.1",\n            f"--http-port={port}",\n            f"--dir={root}",\n            "--verbosity=0",\n            env=environment,\n            stdout=asyncio.subprocess.DEVNULL,\n            stderr=asyncio.subprocess.DEVNULL,\n        )\n        await wait_for_server(process, port)\n        # One migration per server data directory. Do not log out a cloud session on every cell rerun.\n        migrated = root / "cloud-migrated"\n        if not migrated.exists():\n            await migrate_from_cloud(token)\n            migrated.touch(mode=0o600)\n        api = Telegram(token, api_root=f"http://127.0.0.1:{port}", local_files=True)\n        api.server_process = process\n        api.file_directory = root.resolve()\n        yield api\n    except OSError:\n        raise LocalAPIError(\n            "Could not start the local Telegram API server. Run setup again."\n        ) from None\n    finally:\n        if api is not None:\n            await api.close()\n        if process is not None:\n            await stop_server(process)\n', 'vidfetch_bot/media.py': '"""Extraction and media preparation, with lossless delivery through a local Bot API."""\n\nfrom __future__ import annotations\n\nimport asyncio\nimport ipaddress\nimport json\nimport math\nimport os\nimport re\nimport shutil\nimport signal\nimport socket\nimport sys\nfrom collections.abc import Awaitable, Callable\nfrom contextlib import nullcontext\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom urllib.parse import urlsplit\n\nfrom .cookies import CookieError, CookieStore, site_for_url\n\nUPLOAD_LIMIT = 49_000_000  # Below the cloud Bot API\'s 50 MB limit, in bytes.\nTARGET_BYTES = 46_000_000\nMIN_FREE_BYTES = 256_000_000\nAUDIO_BITRATE = 192_000\nStatus = Callable[[str], Awaitable[None]]\n\n\n@dataclass(frozen=True)\nclass StorageBudget:\n    """Account for a hosting quota even when statvfs reports the entire shared disk."""\n\n    root: Path\n    limit: int\n\n    def free(self) -> int:\n        used = 0\n        for path in self.root.rglob("*"):\n            try:\n                if path.is_file():\n                    used += path.stat().st_size\n            except FileNotFoundError:\n                # A concurrent upload or cleanup can remove a temporary file.\n                continue\n        return self.limit - used\n\n\nSTORAGE_BUDGET: StorageBudget | None = None\n\n\nclass MediaError(Exception):\n    """A safe, actionable message that can be displayed in Telegram."""\n\n    def __init__(self, message: str, *, retryable: bool = False):\n        super().__init__(message)\n        self.retryable = retryable\n\n\ndef tool_error(log: str, args: list[str]) -> MediaError:\n    # Use the terminal failure, not a startup warning mentioning cookies or a URL.\n    errors = [line for line in log.splitlines() if line.lstrip().startswith("ERROR:")]\n    detail = (errors[-1] if errors else log).lower()\n    site = site_for_url(args[-1]) if "yt_dlp" in args and args else None\n    if "429" in detail or "too many requests" in detail:\n        return MediaError("This site is rate limiting requests (HTTP 429). Wait before retrying.")\n    if "private subreddit" in detail or "quarantined" in detail:\n        return MediaError("Reddit requires cookies from an account with access to this community.")\n    blocked = any(\n        item in detail\n        for item in (\n            "403",\n            "forbidden",\n            "unable to access the reddit api",\n            "account authentication is required",\n            "impersonate target",\n            "impersonation target",\n        )\n    )\n    if site == "reddit.com" and blocked:\n        return MediaError(\n            "Reddit rejected this session or connection. Import fresh reddit.com cookies from "\n            "Chrome, then send the full post link again. If it still fails, "\n            "Reddit may be rejecting this host\'s network.",\n            retryable=True,\n        )\n    if any(item in detail for item in ("sign in", "login", "authentication", "not a bot")):\n        return MediaError(\n            "This site requires verification or rejected the session. Import or replace its "\n            "cookies, then send the link again. The site may also reject "\n            "the host\'s network."\n        )\n    if blocked:\n        return MediaError(\n            "This site rejected the request (HTTP 403). Refresh its cookies and extractor. "\n            "If it continues, try a host on another network."\n        )\n    if "requested format" in detail:\n        return MediaError("That quality is no longer available. Send the link again to refresh it.")\n    return MediaError(\n        "The media tool could not finish. The link may be unsupported or unavailable."\n    )\n\n\ndef check_disk(folder: Path) -> None:\n    if STORAGE_BUDGET and STORAGE_BUDGET.free() < MIN_FREE_BYTES:\n        raise MediaError(\n            "This host is running out of its storage allowance. Choose a smaller quality or file."\n        )\n    if shutil.disk_usage(folder).free < MIN_FREE_BYTES:\n        raise MediaError(\n            "This session is running out of disk space. Try a smaller quality or restart."\n        )\n\n\n@dataclass(frozen=True)\nclass Quality:\n    height: int | None\n    selector: str\n    size: int | None\n    audio_only: bool = False\n\n    @property\n    def label(self) -> str:\n        return self.label_for(UPLOAD_LIMIT, preserve_quality=False)\n\n    def label_for(self, limit: int, *, preserve_quality: bool) -> str:\n        resolution = (\n            "Audio only"\n            if self.audio_only\n            else (f"{self.height}p" if self.height else "Best available")\n        )\n        size = f"~{format_size(self.size)}" if self.size else "size unknown"\n        marker = (\n            (" · split into files" if preserve_quality else " · compress")\n            if self.size and self.size > limit\n            else ""\n        )\n        return f"{resolution} · {size}{marker}"\n\n\n@dataclass(frozen=True)\nclass Analysis:\n    title: str\n    qualities: tuple[Quality, ...]\n\n\n@dataclass(frozen=True)\nclass Probe:\n    duration: float\n    width: int\n    height: int\n    video_codec: str\n    pixel_format: str\n    audio_codec: str | None\n    fps: float = 30\n    video_index: int = 0\n    audio_index: int | None = None\n\n\n@dataclass(frozen=True)\nclass PreparedVideo:\n    path: Path\n    probe: Probe\n    compressed: bool\n    document: bool = False\n\n\n@dataclass(frozen=True)\nclass AudioProbe:\n    duration: float\n    audio_codec: str\n    bitrate: int\n    audio_index: int\n\n\n@dataclass(frozen=True)\nclass PreparedAudio:\n    path: Path\n    probe: AudioProbe\n    compressed: bool\n    document: bool = False\n\n\ndef number(value: object) -> float:\n    try:\n        result = float(value)\n        return result if math.isfinite(result) and result > 0 else 0\n    except (TypeError, ValueError):\n        return 0\n\n\ndef format_size(size: int) -> str:\n    # Decimal MB matches the upload budget and avoids MiB/MB ambiguity.\n    return f"{size / 1_000_000:.1f} MB"\n\n\ndef estimate_size(fmt: dict, duration: float) -> int | None:\n    for key in ("filesize", "filesize_approx"):\n        if value := number(fmt.get(key)):\n            return math.ceil(value)\n    bitrate = number(fmt.get("tbr"))\n    if not bitrate:\n        bitrate = number(fmt.get("vbr")) + number(fmt.get("abr"))\n    return math.ceil(duration * bitrate * 1000 / 8) if duration and bitrate else None\n\n\ndef is_aac(codec: str | None) -> bool:\n    return codec == "aac" or bool(codec and codec.startswith("mp4a"))\n\n\ndef audio_bitrate(source_bitrate: float) -> int:\n    return min(AUDIO_BITRATE, max(32_000, int(source_bitrate))) if source_bitrate else AUDIO_BITRATE\n\n\ndef estimate_audio_size(\n    fmt: dict, duration: float, *, preserve_quality: bool = False\n) -> int | None:\n    """Estimate the delivered audio, never the full video used as a fallback source."""\n    separate = fmt.get("vcodec") == "none"\n    if preserve_quality:\n        if separate:\n            return estimate_size(fmt, duration)\n        bitrate = number(fmt.get("abr"))\n        return math.ceil(duration * bitrate * 1000 / 8 * 1.02) if duration and bitrate else None\n    if is_aac(fmt.get("acodec")) and separate:\n        return estimate_size(fmt, duration)\n    bitrate = number(fmt.get("abr")) or (number(fmt.get("tbr")) if separate else 0)\n    if not duration:\n        return None\n    if is_aac(fmt.get("acodec")):\n        return math.ceil(duration * bitrate * 1000 / 8 * 1.02) if bitrate else None\n    # Non-AAC sources use one known conversion bitrate, so the menu does not\n    # depend on source bitrate fields that ffprobe may omit (notably for Opus).\n    return math.ceil(duration * AUDIO_BITRATE / 8 * 1.02)\n\n\ndef qualities_from_info(info: dict, *, preserve_quality: bool = False) -> tuple[Quality, ...]:\n    """Ask yt-dlp to select formats so displayed sizes match the download selector."""\n    from yt_dlp import YoutubeDL\n\n    formats = [f for f in info.get("formats", [info]) if not f.get("has_drm")]\n    heights = sorted(\n        {int(h) for f in formats if f.get("vcodec") != "none" and (h := number(f.get("height")))},\n        reverse=True,\n    )\n    duration = number(info.get("duration"))\n    result = []\n    host = (urlsplit(info.get("webpage_url") or "").hostname or "").lower()\n    direct_mp4_first = host == "pornhub.com" or host.endswith(".pornhub.com")\n    # The extractor returns formats sorted by yt-dlp\'s configured preference.\n    context = {\n        "formats": formats,\n        "has_merged_format": any(\n            f.get("vcodec") != "none" and f.get("acodec") != "none" for f in formats\n        ),\n        "incomplete_formats": not any(f.get("vcodec") == "none" for f in formats),\n    }\n    with YoutubeDL({"quiet": True, "no_warnings": True}) as ydl:\n        for height in heights or [None]:\n            ceiling = f"[height<={height}]" if height else ""\n            selector = f"bv{ceiling}+ba/b{ceiling}/bv{ceiling}"\n            if direct_mp4_first:\n                selector = f"b[ext=mp4]{ceiling}/" + selector\n            selected = list(ydl.build_format_selector(selector)(context))\n            if not selected:\n                continue\n            chosen = selected[-1]\n            actual_height = int(number(chosen.get("height"))) or None\n            if any(q.height == actual_height for q in result):\n                continue\n            streams = chosen.get("requested_formats") or [chosen]\n            sizes = [estimate_size(stream, duration) for stream in streams]\n            size = sum(sizes) if all(s is not None for s in sizes) else None\n            # Use the exact selected IDs on re-extraction, rather than guessing again.\n            identifiers = [str(stream.get("format_id", "")) for stream in streams]\n            if not all(re.fullmatch(r"[\\w.-]+", value, flags=re.ASCII) for value in identifiers):\n                continue\n            result.append(Quality(actual_height, "+".join(identifiers), size))\n        # Download a standalone audio stream if available. Otherwise take the\n        # smallest-preference combined format, avoiding a full-resolution video download.\n        audio_formats = list(ydl.build_format_selector("ba")(context))\n        audio = (\n            audio_formats[-1]\n            if audio_formats\n            else next(\n                (f for f in formats if f.get("acodec") != "none" and f.get("vcodec") != "none"),\n                None,\n            )\n        )\n        if audio and audio.get("acodec") != "none":\n            identifier = str(audio.get("format_id", ""))\n            if re.fullmatch(r"[\\w.-]+", identifier, flags=re.ASCII):\n                result.append(\n                    Quality(\n                        None,\n                        identifier,\n                        estimate_audio_size(audio, duration, preserve_quality=preserve_quality),\n                        audio_only=True,\n                    )\n                )\n    if not result:\n        raise MediaError("No downloadable video or audio was found for this link.")\n    return tuple(result)\n\n\nasync def validate_url(url: str) -> str:\n    try:\n        parts = urlsplit(url)\n        host = parts.hostname\n        port = parts.port\n        if parts.scheme not in {"http", "https"} or not host or parts.username or parts.password:\n            raise ValueError\n        if port not in {None, 80, 443} or len(url) > 4096:\n            raise ValueError\n        addresses = await asyncio.get_running_loop().getaddrinfo(\n            host, port or (443 if parts.scheme == "https" else 80), type=socket.SOCK_STREAM\n        )\n        if not addresses or any(\n            not ipaddress.ip_address(item[4][0]).is_global for item in addresses\n        ):\n            raise ValueError\n    except (ValueError, OSError):\n        raise MediaError("Send a public http:// or https:// video link.") from None\n    return url\n\n\nasync def command(\n    args: list[str], *, timeout: int, folder: Path, download_guard: bool = False\n) -> str:\n    """Cancel/timeout always reaps the child before the caller deletes its files."""\n    if download_guard:\n        check_disk(folder)\n    log_path = folder / "process.stderr"\n    with log_path.open("wb") as log:\n        process = await asyncio.create_subprocess_exec(\n            *args, stdout=asyncio.subprocess.PIPE, stderr=log, start_new_session=os.name != "nt"\n        )\n        communicate = asyncio.create_task(process.communicate())\n        try:\n            async with asyncio.timeout(timeout):\n                while not communicate.done():\n                    await asyncio.wait({communicate}, timeout=1)\n                    if download_guard:\n                        check_disk(folder)\n                output, _ = await communicate\n        except TimeoutError:\n            raise MediaError(\n                "This operation took too long. Try a shorter video or lower quality."\n            ) from None\n        finally:\n            if process.returncode is None:\n                if os.name == "nt":\n                    # yt-dlp can itself spawn FFmpeg; reap that child as well.\n                    killer = await asyncio.create_subprocess_exec(\n                        "taskkill",\n                        "/PID",\n                        str(process.pid),\n                        "/T",\n                        "/F",\n                        stdout=asyncio.subprocess.DEVNULL,\n                        stderr=asyncio.subprocess.DEVNULL,\n                    )\n                    await killer.wait()\n                else:\n                    try:\n                        os.killpg(process.pid, signal.SIGTERM)\n                    except ProcessLookupError:\n                        pass\n                try:\n                    await asyncio.wait_for(process.wait(), timeout=5)\n                except TimeoutError:\n                    try:\n                        if os.name == "nt":\n                            process.kill()\n                        else:\n                            os.killpg(process.pid, signal.SIGKILL)\n                    except ProcessLookupError:\n                        pass\n                    await process.wait()\n            if os.name != "nt":\n                # The parent may exit before a Deno/FFmpeg descendant releases stdout.\n                try:\n                    os.killpg(process.pid, signal.SIGKILL)\n                except ProcessLookupError:\n                    pass\n            if not communicate.done():\n                communicate.cancel()\n            await asyncio.gather(communicate, return_exceptions=True)\n    if process.returncode:\n        raise tool_error(log_path.read_text(encoding="utf-8", errors="replace"), args)\n    if len(output) > 16_000_000:\n        raise MediaError("The site returned too much metadata. Send a single-video link.")\n    return output.decode("utf-8", errors="replace").strip()\n\n\ndef extractor_args(*, cookie: Path | None = None, chrome: bool = False) -> list[str]:\n    import deno\n\n    args = [\n        sys.executable,\n        "-m",\n        "yt_dlp",\n        "--ignore-config",\n        "--no-cache-dir",\n        "--no-playlist",\n        "--playlist-items",\n        "1",\n        "--socket-timeout",\n        "25",\n        "--retries",\n        "3",\n        "--fragment-retries",\n        "3",\n        "--abort-on-unavailable-fragments",\n        "--no-progress",\n        "--no-warnings",\n        "--js-runtimes",\n        f"deno:{deno.find_deno_bin()}",\n        "--format-sort",\n        "res,vcodec:h264,acodec:aac",\n    ]\n    if cookie:\n        args += ["--cookies", str(cookie)]\n    if chrome:\n        args += ["--impersonate", "chrome"]\n    return args\n\n\nasync def extract(\n    url: str,\n    arguments: list[str],\n    folder: Path,\n    *,\n    timeout: int,\n    cookie_store: CookieStore | None = None,\n    download_guard: bool = False,\n) -> str:\n    reddit = site_for_url(url) == "reddit.com"\n    plugins = folder / "extractor-plugins"\n    plugin = plugins / "vidfetch" / "yt_dlp_plugins" / "extractor" / "reddit_public.py"\n    plugin.parent.mkdir(parents=True, exist_ok=True)\n    shutil.copyfile(Path(__file__).with_name("reddit_public.py"), plugin)\n    for chrome in (True, False) if reddit else (False,):\n        context = cookie_store.snapshot_for(url, folder) if cookie_store else nullcontext(None)\n        try:\n            with context as cookie:\n                restart = (\n                    ["--force-overwrites", "--no-continue"] if reddit and download_guard else []\n                )\n                return await command(\n                    extractor_args(cookie=cookie, chrome=chrome)\n                    + ["--plugin-dirs", str(plugins)]\n                    + restart\n                    + arguments\n                    + ["--", url],\n                    timeout=timeout,\n                    folder=folder,\n                    download_guard=download_guard,\n                )\n        except CookieError as error:\n            raise MediaError(str(error)) from None\n        except MediaError as error:\n            if not chrome or not error.retryable:\n                raise\n    raise AssertionError("No extraction attempt")\n\n\nasync def analyze(\n    url: str,\n    folder: Path,\n    *,\n    preserve_quality: bool = False,\n    cookie_store: CookieStore | None = None,\n) -> Analysis:\n    await validate_url(url)\n    raw = await extract(\n        url,\n        ["--dump-single-json", "--skip-download"],\n        folder,\n        timeout=180,\n        cookie_store=cookie_store,\n    )\n    try:\n        info = json.loads(raw)\n        if info.get("_type") in {"playlist", "multi_video"} or info.get("entries") is not None:\n            raise MediaError("Send a single-video link rather than a playlist.")\n        if info.get("is_live") or info.get("live_status") == "is_live":\n            raise MediaError("Live streams are not supported. Send a finished video.")\n        return Analysis(\n            str(info.get("title") or "Video")[:200],\n            qualities_from_info(info, preserve_quality=preserve_quality),\n        )\n    except (ValueError, TypeError, KeyError):\n        raise MediaError("The site did not return usable video information.") from None\n\n\nasync def download(\n    url: str, quality: Quality, folder: Path, *, cookie_store: CookieStore | None = None\n) -> Path:\n    # Recheck the initial host when a button is used later in the session.\n    await validate_url(url)\n    raw = await extract(\n        url,\n        [\n            "--format",\n            quality.selector,\n            "--merge-output-format",\n            "mkv",\n            "--output",\n            str(folder / "source.%(ext)s"),\n            "--print",\n            "after_move:%(filepath)j",\n        ],\n        folder,\n        timeout=7200,\n        cookie_store=cookie_store,\n        download_guard=True,\n    )\n    try:\n        path = Path(json.loads(raw.splitlines()[-1])).resolve()\n        if path.parent != folder.resolve() or not path.is_file() or not path.stat().st_size:\n            raise ValueError\n        return path\n    except (ValueError, IndexError, TypeError):\n        kind = "audio" if quality.audio_only else "video"\n        raise MediaError(f"The extractor did not produce a completed {kind} file.") from None\n\n\nasync def probe_audio(path: Path, *, audio_only: bool = False) -> AudioProbe:\n    raw = await command(\n        ["ffprobe", "-v", "error", "-show_streams", "-show_format", "-of", "json", str(path)],\n        timeout=30,\n        folder=path.parent,\n    )\n    try:\n        info = json.loads(raw)\n        streams = info.get("streams", [])\n        audio = next(s for s in streams if s.get("codec_type") == "audio")\n        if audio_only and any(s.get("codec_type") == "video" for s in streams):\n            raise ValueError\n        duration = number(audio.get("duration")) or number(info.get("format", {}).get("duration"))\n        if not duration or not audio.get("codec_name"):\n            raise ValueError\n        return AudioProbe(\n            duration, audio["codec_name"], int(number(audio.get("bit_rate"))), int(audio["index"])\n        )\n    except (ValueError, KeyError, StopIteration, TypeError):\n        raise MediaError(\n            "This file does not contain a valid audio track. Try another link."\n        ) from None\n\n\ndef check_audio_duration(original: AudioProbe, final: AudioProbe) -> None:\n    if abs(final.duration - original.duration) > max(2, original.duration * 0.01):\n        raise MediaError("The prepared audio was incomplete. Please try a shorter clip.")\n\n\nasync def prepare_audio(\n    source: Path,\n    status: Status,\n    *,\n    limit: int = UPLOAD_LIMIT,\n    target: int = TARGET_BYTES,\n    preserve_quality: bool = False,\n) -> PreparedAudio:\n    info = await probe_audio(source)\n    if preserve_quality:\n        extension = {"aac": ".m4a", "mp3": ".mp3"}.get(info.audio_codec, ".mka")\n        output = source.parent / ("VidFetch" + extension)\n        await status("Extracting the original audio without reducing quality…")\n        check_disk(source.parent)\n        await command(\n            [\n                "ffmpeg",\n                "-hide_banner",\n                "-loglevel",\n                "error",\n                "-nostdin",\n                "-y",\n                "-i",\n                str(source),\n                "-map",\n                f"0:{info.audio_index}",\n                "-vn",\n                "-sn",\n                "-dn",\n                "-c:a",\n                "copy",\n                str(output),\n            ],\n            timeout=7200,\n            folder=source.parent,\n            download_guard=True,\n        )\n        final = await probe_audio(output, audio_only=True)\n        check_audio_duration(info, final)\n        return PreparedAudio(output, final, False, document=extension == ".mka")\n    output = source.parent / "telegram.m4a"\n    base = [\n        "ffmpeg",\n        "-hide_banner",\n        "-loglevel",\n        "error",\n        "-nostdin",\n        "-y",\n        "-i",\n        str(source),\n        "-map",\n        f"0:{info.audio_index}",\n        "-vn",\n        "-sn",\n        "-dn",\n    ]\n    await status("Extracting audio for Telegram…")\n    compressed = False\n    if info.audio_codec == "aac":\n        # Copying only the audio also avoids needless compression when the source\n        # video is large but its AAC track already fits Telegram\'s upload budget.\n        await command(\n            base + ["-c:a", "copy", "-movflags", "+faststart", str(output)],\n            timeout=300,\n            folder=source.parent,\n        )\n        final = await probe_audio(output, audio_only=True)\n        check_audio_duration(info, final)\n        if output.stat().st_size <= limit:\n            return PreparedAudio(output, final, False)\n        compressed = True\n        await status("Compressing audio to fit Telegram\'s upload limit…")\n    preferred_rate = audio_bitrate(info.bitrate) if info.audio_codec == "aac" else AUDIO_BITRATE\n    for attempt in range(3):\n        budget_rate = int(min(target, limit * 0.94) * 8 / info.duration * 0.95 * 0.8**attempt)\n        rate = min(preferred_rate, budget_rate)\n        if rate < 16_000:\n            raise MediaError(\n                "This audio is too long to fit a single Telegram upload. Try a shorter clip."\n            )\n        if rate < preferred_rate or attempt:\n            compressed = True\n            await status("Compressing audio to fit Telegram\'s upload limit…")\n        await command(\n            base\n            + [\n                "-c:a",\n                "aac",\n                "-b:a",\n                str(rate),\n                "-ac",\n                "1" if rate < 64_000 else "2",\n                "-movflags",\n                "+faststart",\n                str(output),\n            ],\n            timeout=1800,\n            folder=source.parent,\n        )\n        final = await probe_audio(output, audio_only=True)\n        check_audio_duration(info, final)\n        if output.stat().st_size <= limit:\n            return PreparedAudio(output, final, compressed)\n    raise MediaError("This audio could not fit Telegram\'s limit. Try a shorter clip.")\n\n\nasync def probe(path: Path) -> Probe:\n    raw = await command(\n        ["ffprobe", "-v", "error", "-show_streams", "-show_format", "-of", "json", str(path)],\n        timeout=30,\n        folder=path.parent,\n    )\n    try:\n        info = json.loads(raw)\n        streams = info.get("streams", [])\n        video = next(\n            s\n            for s in streams\n            if s.get("codec_type") == "video" and not s.get("disposition", {}).get("attached_pic")\n        )\n        audio = next((s for s in streams if s.get("codec_type") == "audio"), None)\n        duration = number(info.get("format", {}).get("duration")) or number(video.get("duration"))\n        width, height = int(video["width"]), int(video["height"])\n        rotation = next(\n            (s.get("rotation", 0) for s in video.get("side_data_list", []) if "rotation" in s), 0\n        )\n        if abs(float(rotation)) % 180 == 90:\n            width, height = height, width\n        fps_parts = video.get("avg_frame_rate", "30/1").split("/")\n        fps = number(fps_parts[0]) / (number(fps_parts[-1]) or 1)\n        if not duration or width < 2 or height < 2:\n            raise ValueError\n        return Probe(\n            duration,\n            width,\n            height,\n            video.get("codec_name", ""),\n            video.get("pix_fmt", ""),\n            audio.get("codec_name", "") if audio else None,\n            fps or 30,\n            int(video.get("index", 0)),\n            int(audio["index"]) if audio else None,\n        )\n    except (ValueError, KeyError, StopIteration, TypeError):\n        raise MediaError("The downloaded file does not contain a valid, finished video.") from None\n\n\ndef encoding_plan(info: Probe, target_bytes: int, attempt: int = 0) -> tuple[int, int, str]:\n    total_rate = int(target_bytes * 8 / info.duration * 0.94 * 0.82**attempt)\n    audio_rate = (\n        (96_000 if total_rate >= 400_000 else 48_000 if total_rate >= 120_000 else 16_000)\n        if info.audio_codec\n        else 0\n    )\n    video_rate = total_rate - audio_rate\n    if video_rate < 8_000:\n        raise MediaError(\n            "This video is too long to fit a single Telegram upload. Try a shorter clip."\n        )\n    fps = min(info.fps, 30 if video_rate >= 250_000 else 20 if video_rate >= 80_000 else 10)\n    scale = min(\n        1,\n        1920 / max(info.width, info.height),\n        math.sqrt(video_rate / (max(fps, 1) * 0.07 * info.width * info.height)),\n    )\n    width = max(2, int(info.width * scale) // 2 * 2)\n    height = max(2, int(info.height * scale) // 2 * 2)\n    return video_rate, audio_rate, f"scale={width}:{height},setsar=1,fps={fps:.3f}"\n\n\nasync def prepare_video(\n    source: Path,\n    status: Status,\n    *,\n    limit: int = UPLOAD_LIMIT,\n    target: int = TARGET_BYTES,\n    preserve_quality: bool = False,\n) -> PreparedVideo:\n    info = await probe(source)\n    if preserve_quality:\n        # An original codec that cannot play inline is delivered as a document, not transcoded.\n        compatible = (\n            info.video_codec == "h264"\n            and info.pixel_format == "yuv420p"\n            and info.audio_codec in {None, "aac"}\n        )\n        if not compatible or source.suffix.lower() == ".mp4":\n            return PreparedVideo(source, info, False, document=not compatible)\n        output = source.parent / "VidFetch.mp4"\n        await status("Preparing the original video without reducing quality…")\n        check_disk(source.parent)\n        await command(\n            [\n                "ffmpeg",\n                "-hide_banner",\n                "-loglevel",\n                "error",\n                "-nostdin",\n                "-y",\n                "-i",\n                str(source),\n                "-map",\n                f"0:{info.video_index}",\n                "-map",\n                "0:a:0?",\n                "-c",\n                "copy",\n                "-sn",\n                "-dn",\n                "-movflags",\n                "+faststart",\n                str(output),\n            ],\n            timeout=7200,\n            folder=source.parent,\n            download_guard=True,\n        )\n        final = await probe(output)\n        if abs(final.duration - info.duration) > max(2, info.duration * 0.01) or (\n            info.audio_codec and not final.audio_codec\n        ):\n            raise MediaError("The prepared video was incomplete. Try the original format again.")\n        return PreparedVideo(output, final, False)\n    output = source.parent / "telegram.mp4"\n    base = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", "-y", "-i", str(source)]\n    if (\n        source.stat().st_size <= limit\n        and info.video_codec == "h264"\n        and info.pixel_format == "yuv420p"\n    ):\n        await status("Preparing your video for Telegram…")\n        await command(\n            base\n            + [\n                "-map",\n                f"0:{info.video_index}",\n                "-map",\n                f"0:{info.audio_index}?" if info.audio_index is not None else "0:a:0?",\n                "-sn",\n                "-dn",\n                "-c:v",\n                "copy",\n                "-c:a",\n                "copy" if info.audio_codec in {None, "aac"} else "aac",\n                "-movflags",\n                "+faststart",\n                str(output),\n            ],\n            timeout=300,\n            folder=source.parent,\n        )\n        final = await probe(output)\n        if output.stat().st_size <= limit:\n            return PreparedVideo(output, final, False)\n    await status("Compressing for Telegram… Quality may be reduced to fit the upload limit.")\n    for attempt in range(3):\n        rate, audio_rate, filters = encoding_plan(info, min(target, int(limit * 0.94)), attempt)\n        video_args = [\n            "-map",\n            f"0:{info.video_index}",\n            "-vf",\n            filters,\n            "-c:v",\n            "libx264",\n            "-preset",\n            "fast",\n            "-pix_fmt",\n            "yuv420p",\n            "-b:v",\n            str(rate),\n            "-threads",\n            "2",\n            "-passlogfile",\n            str(source.parent / "encode-pass"),\n        ]\n        await command(\n            base + video_args + ["-pass", "1", "-an", "-sn", "-dn", "-f", "null", os.devnull],\n            timeout=1800,\n            folder=source.parent,\n        )\n        audio_args = (\n            [\n                "-map",\n                f"0:{info.audio_index}?",\n                "-c:a",\n                "aac",\n                "-b:a",\n                str(audio_rate),\n                "-ac",\n                "1" if audio_rate < 48_000 else "2",\n            ]\n            if info.audio_codec\n            else ["-an"]\n        )\n        await command(\n            base\n            + video_args\n            + ["-pass", "2"]\n            + audio_args\n            + ["-sn", "-dn", "-movflags", "+faststart", "-metadata:s:v:0", "rotate=0", str(output)],\n            timeout=1800,\n            folder=source.parent,\n        )\n        final = await probe(output)\n        if output.stat().st_size <= limit:\n            # A complete clip must survive compression, including any original audio.\n            if abs(final.duration - info.duration) > max(2, info.duration * 0.01):\n                raise MediaError("The compressed video was incomplete. Please try a shorter clip.")\n            if info.audio_codec and not final.audio_codec:\n                raise MediaError("Audio was missing after compression. Please try another quality.")\n            return PreparedVideo(output, final, True)\n    raise MediaError("This video could not fit Telegram\'s limit. Try a shorter clip.")\n', 'vidfetch_bot/personal.py': '"""Private-chat access management, explicit cookie delegation and custom website imports."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport re\nimport secrets\nimport time\nfrom dataclasses import dataclass, field\n\nfrom . import media\nfrom .access import AccessStore, valid_id\nfrom .cookie_routing import RoutedCookieStore\nfrom .cookies import (\n    MAX_COOKIE_BYTES,\n    MAX_SITES,\n    CookieError,\n    UserCookieStores,\n    normalize_site,\n    scoped_cookies,\n)\n\nSITES = {"youtube": "youtube.com", "reddit": "reddit.com"}\nLABELS = {"youtube.com": "YouTube", "reddit.com": "Reddit"}\nPAGE_SIZE = 4\nOWNER_HELP = (\n    "\\n\\nOwner controls:\\n/allow ID — approve users (multiple IDs may be separated by spaces)\\n"\n    "/deny ID — revoke access, cancel requests and forget their cookies\\n"\n    "/users — list approved users and cookie-sharing status\\n"\n    "/sharecookies ID [domain|all] — let an approved user use your cookies\\n"\n    "/unsharecookies ID [domain|all] — remove cookie sharing"\n)\n\n\ndef site_label(site: str) -> str:\n    return LABELS.get(site, site if len(site) <= 40 else site[:37] + "...")\n\n\ndef site_selector(site: str) -> str:\n    return {"youtube.com": "youtube", "reddit.com": "reddit"}.get(site, "s" + hashlib.sha256(site.encode()).hexdigest()[:16])\n\n\n@dataclass\nclass CookieMenu:\n    owner: tuple[int, int]\n    expires: float\n    confirmation_site: str | None = None\n    sites: dict[str, str] = field(default_factory=dict)\n    page: int = 0\n\n\nclass PersonalControls:\n    def __init__(self, bot, access: AccessStore, stores: UserCookieStores):\n        self.bot = bot\n        self.api = bot.api\n        self.access = access\n        self.stores = stores\n        self.menus: dict[str, CookieMenu] = {}\n        self.pending: dict[tuple[int, int], tuple[str, float]] = {}\n        self.pending_domain: dict[tuple[int, int], float] = {}\n\n    def expire(self, owner: tuple[int, int] | None = None) -> None:\n        now = time.monotonic()\n        self.menus = {key: menu for key, menu in self.menus.items() if menu.expires > now and menu.owner != owner}\n        self.pending = {key: value for key, value in self.pending.items() if value[1] > now}\n        self.pending_domain = {key: expires for key, expires in self.pending_domain.items() if expires > now}\n        while len(self.menus) >= 200:\n            self.menus.pop(next(iter(self.menus)))\n\n    def forget_controls(self, user_id: int) -> None:\n        self.menus = {key: menu for key, menu in self.menus.items() if menu.owner[1] != user_id}\n        self.pending = {owner: value for owner, value in self.pending.items() if owner[1] != user_id}\n        self.pending_domain = {owner: expires for owner, expires in self.pending_domain.items() if owner[1] != user_id}\n\n    def menu_key(self, owner: tuple[int, int], confirmation_site: str | None = None, *, sites=(), page=0) -> str:\n        self.expire(owner)\n        key = secrets.token_urlsafe(8)\n        values = tuple(sites) or ((confirmation_site,) if confirmation_site else tuple(SITES.values()))\n        self.menus[key] = CookieMenu(owner, time.monotonic() + 600, confirmation_site, {site_selector(site): site for site in values}, page)\n        return key\n\n    async def show(self, owner: tuple[int, int], page: int = 0) -> None:\n        if not self.access.allows(owner[1]):\n            return\n        self.pending_domain.pop(owner, None)\n        store = self.stores.for_user(owner[1])\n        routing = RoutedCookieStore(self.access, self.stores, owner[1])\n        sites = tuple(dict.fromkeys((*SITES.values(), *store.sites(), *routing.shared_sites())))\n        pages = max(1, (len(sites) + PAGE_SIZE - 1) // PAGE_SIZE)\n        page = max(0, min(page, pages - 1))\n        visible = sites[page * PAGE_SIZE:(page + 1) * PAGE_SIZE]\n        key = self.menu_key(owner, sites=visible, page=page)\n        buttons = []\n        for site in visible:\n            selector, label = site_selector(site), site_label(site)\n            buttons.append([{"text": f"Import / replace {label}", "callback_data": f"c:{key}:import:{selector}"}])\n            if store.has_site(site):\n                action = "disable" if store.is_enabled(site) else "enable"\n                buttons.append([\n                    {"text": f"{action.title()} {label}", "callback_data": f"c:{key}:{action}:{selector}"},\n                    {"text": f"Delete {label}", "callback_data": f"c:{key}:delete:{selector}"},\n                ])\n        buttons.append([{"text": "Add another site", "callback_data": f"c:{key}:add:new"}])\n        navigation = []\n        if page:\n            navigation.append({"text": "Previous sites", "callback_data": f"c:{key}:page:{page - 1}"})\n        if page + 1 < pages:\n            navigation.append({"text": "More sites", "callback_data": f"c:{key}:page:{page + 1}"})\n        if navigation:\n            buttons.append(navigation)\n        await self.api.message(\n            owner[0], f"Your personal cookies · page {page + 1}/{pages}\\n\\nYour saved exports:\\n" + store.status(visible)\n            + "\\n\\n" + routing.summary(visible)\n            + "\\n\\nUse /cookies example.com to import any website\'s cookies. These buttons manage only your own exports. "\n            "Cookies pass through Telegram and are stored on this bot\'s host; host administrators can access server files. "\n            "Only upload your own sessions. /cancel cancels an import.",\n            keyboard=buttons,\n        )\n\n    async def cancel_dependents(self, provider: int) -> None:\n        for recipient in self.access.recipients_for_owner(provider):\n            await self.bot.cancel_user(recipient)\n\n    async def start_import(self, owner: tuple[int, int], value: str) -> None:\n        try:\n            site = normalize_site(value)\n            store = self.stores.for_user(owner[1])\n            if not store.has_site(site) and len(store.sites()) >= MAX_SITES:\n                raise CookieError(f"You can save cookies for {MAX_SITES} sites. Delete an old site first.")\n        except CookieError as error:\n            await self.api.message(owner[0], str(error))\n            return\n        await self.bot.cancel_user(owner[1])\n        await self.cancel_dependents(owner[1])\n        self.pending[owner] = (site, time.monotonic() + 600)\n        sharing = " Users you have given owner-cookie access may also use this import." if owner[1] in self.access.owners else ""\n        await self.api.message(\n            owner[0], f"Send your {site} Netscape cookies.txt export as a document within 10 minutes (maximum 2 MB). "\n            "It will pass through Telegram and be saved as your personal export on this host." + sharing\n            + " Only upload your own session. You may remove the upload message afterward. Use /cancel to stop."\n        )\n\n    async def domain_message(self, owner: tuple[int, int], text: str) -> bool:\n        self.expire()\n        if owner not in self.pending_domain or not text or text.startswith("/"):\n            return False\n        await self.start_import(owner, text)\n        return True\n\n    async def admin(self, owner: tuple[int, int], text: str) -> bool:\n        parts = text.split(maxsplit=1)\n        if not parts:\n            return False\n        command = parts[0].split("@")[0].lower()\n        if command not in {"/allow", "/deny", "/users", "/sharecookies", "/unsharecookies"}:\n            return False\n        if owner[1] not in self.access.owners:\n            await self.api.message(owner[0], "Only the bot owner can manage access and cookie sharing.")\n            return True\n        try:\n            if command == "/users":\n                page = int(parts[1]) if len(parts) > 1 else 1\n                users = sorted(self.access.users)\n                pages = max(1, (len(users) + 19) // 20)\n                if not 1 <= page <= pages:\n                    raise ValueError(f"Use /users with a page number from 1 to {pages}.")\n                rows = ["Owners: " + ", ".join(map(str, sorted(self.access.owners))), f"Approved users · page {page}/{pages}:"]\n                for user in users[(page - 1) * 20:page * 20]:\n                    grants = self.access.shares.get(user, {})\n                    label = "owner cookie set" if "*" in grants else f"{sum(bool(v) for v in grants.values())} shared sites"\n                    rows.append(str(user) + (" · " + label if grants else " · personal cookies"))\n                if not users:\n                    rows.append("None yet. Use /allow ID.")\n                if page < pages:\n                    rows.append(f"Next page: /users {page + 1}")\n                await self.api.message(owner[0], "\\n".join(rows))\n                return True\n            if command in {"/sharecookies", "/unsharecookies"}:\n                arguments = parts[1].split() if len(parts) > 1 else []\n                if len(arguments) not in {1, 2} or not re.fullmatch(r"[0-9]{1,19}", arguments[0]):\n                    raise ValueError(f"Use {command} USER_ID [domain|all].")\n                recipient = int(arguments[0])\n                if not valid_id(recipient):\n                    raise ValueError("Use a positive Telegram user ID.")\n                site = arguments[1] if len(arguments) > 1 else "all"\n                site = "*" if site.lower() in {"*", "all"} else normalize_site(site)\n                if command == "/sharecookies":\n                    self.access.share_cookies(recipient, owner[1], site)\n                    detail = "your entire cookie set, including future imports" if site == "*" else f"your cookies for {site}"\n                    result = f"{recipient} can now use {detail}. The cookie files are not sent to them."\n                else:\n                    self.access.unshare_cookies(recipient, site)\n                    result = f"Cookie sharing removed for {recipient}" + ("." if site == "*" else f" on {site}.")\n                await self.bot.cancel_user(recipient)\n                await self.api.message(owner[0], result + " Existing requests were canceled; new requests use the updated setting.")\n                return True\n            arguments = re.split(r"[\\s,]+", parts[1].strip()) if len(parts) > 1 else []\n            if not arguments or any(not re.fullmatch(r"[0-9]{1,19}", value) for value in arguments):\n                raise ValueError(f"Use {command} followed by numeric Telegram user IDs.")\n            users = {int(value) for value in arguments}\n            if not all(valid_id(value) for value in users):\n                raise ValueError("Telegram user IDs must be positive integers.")\n            if command == "/allow":\n                self.access.allow(users)\n                result = "Approved: "\n            else:\n                self.access.deny(users)\n                for user_id in users:\n                    await self.bot.cancel_user(user_id)\n                    try:\n                        self.stores.forget(user_id)\n                    except OSError:\n                        await self.api.message(owner[0], "Access and cookie sharing are revoked, but export cleanup could not finish. Check server storage permissions.")\n                        return True\n                result = "Access/sharing revoked and personal exports forgotten: "\n            details = ", ".join(map(str, sorted(users))) if len(users) <= 10 else f"{len(users)} users. Use /users for the list."\n            await self.api.message(owner[0], result + details)\n        except ValueError as error:\n            await self.api.message(owner[0], str(error))\n        except OSError:\n            await self.api.message(owner[0], "Could not save changes. Check server storage and try again.")\n        return True\n\n    async def callback(self, query: dict, owner: tuple[int, int]) -> None:\n        self.expire()\n        match = re.fullmatch(r"c:([A-Za-z0-9_-]+):(import|enable|disable|delete|confirm|cancel|add|page):([A-Za-z0-9_-]{1,20})", query.get("data", ""))\n        menu = self.menus.get(match[1]) if match else None\n        action = match[2] if match else ""\n        selector = match[3] if match else ""\n        site = menu.sites.get(selector) if menu else None\n        special = (action == "add" and selector == "new") or (action == "page" and selector.isdecimal())\n        if (\n            not menu or menu.owner != owner or (not site and not special)\n            or (action == "confirm" and menu.confirmation_site != site)\n            or (menu.confirmation_site is not None and (action not in {"confirm", "cancel"} or menu.confirmation_site != site))\n        ):\n            await self.api.acknowledge(query["id"], "This cookie menu expired. Use /cookies.")\n            return\n        self.menus.pop(match[1], None)\n        await self.api.acknowledge(query["id"])\n        if action == "page":\n            await self.show(owner, int(selector))\n            return\n        if action == "add":\n            await self.bot.cancel_user(owner[1])\n            self.pending_domain[owner] = time.monotonic() + 600\n            await self.api.message(owner[0], "Send a website domain, such as youtube.com or example.com. Then I\'ll ask for its Netscape cookie export. Use /cancel to stop.")\n            return\n        if action == "import":\n            await self.start_import(owner, site)\n            return\n        if action == "delete":\n            key = self.menu_key(owner, site)\n            await self.api.message(owner[0], f"Delete your saved cookies for {site}?", keyboard=[[\n                {"text": "Delete my cookies", "callback_data": f"c:{key}:confirm:{site_selector(site)}"},\n                {"text": "Keep them", "callback_data": f"c:{key}:cancel:{site_selector(site)}"},\n            ]])\n            return\n        if action == "cancel":\n            await self.show(owner, menu.page)\n            return\n        await self.bot.cancel_user(owner[1])\n        try:\n            await self.cancel_dependents(owner[1])\n            store = self.stores.for_user(owner[1])\n            if action == "confirm":\n                store.delete(site)\n            else:\n                store.set_enabled(site, action == "enable")\n            await self.show(owner, menu.page)\n        except (CookieError, OSError):\n            await self.api.message(owner[0], "Could not update your cookies. Use /cookies and try again.")\n\n    async def document(self, message: dict, owner: tuple[int, int]) -> None:\n        self.expire()\n        pending = self.pending.get(owner)\n        if not pending:\n            await self.api.message(owner[0], "Use /cookies DOMAIN or choose Import before sending a cookie file.")\n            return\n        document = message["document"]\n        size, file_id = document.get("file_size"), document.get("file_id")\n        if type(size) is not int or not 0 < size <= MAX_COOKIE_BYTES or not isinstance(file_id, str) or not file_id:\n            await self.api.message(owner[0], "Send a Netscape cookie export with a known size under 2 MB.")\n            return\n        site = pending[0]\n        await self.cancel_dependents(owner[1])\n\n        async def import_cookie(status, folder):\n            media.check_disk(folder)\n            await status("Importing your cookie export…")\n            data = await self.api.read_document(file_id, max_bytes=MAX_COOKIE_BYTES)\n            try:\n                filtered = scoped_cookies(data, site).encode("utf-8")\n            except CookieError as error:\n                await status(str(error))\n                return\n            finally:\n                del data\n            if not self.access.allows(owner[1]):\n                return\n            try:\n                await self.cancel_dependents(owner[1])\n                self.stores.for_user(owner[1]).import_bytes(site, filtered)\n            except CookieError as error:\n                await status(str(error))\n                return\n            finally:\n                del filtered\n            await status(f"Your {site} cookies were imported. Any explicit owner-sharing permissions also apply to this export.")\n            await self.show(owner)\n\n        if await self.bot.schedule(owner, import_cookie):\n            self.pending.pop(owner, None)\n', 'vidfetch_bot/reddit_public.py': '"""Use media exposed by the requested Reddit post when its JSON endpoint rejects access."""\n\nimport json\nfrom html.parser import HTMLParser\nfrom urllib.parse import urlsplit\n\nfrom yt_dlp.extractor.reddit import RedditIE\nfrom yt_dlp.utils import ExtractorError\n\n\nclass _PublicPost(HTMLParser):\n    def __init__(self, post_id):\n        super().__init__(convert_charrefs=True)\n        self.post_id = "t3_" + post_id\n        self.depth = 0\n        self.post = None\n        self.player = None\n\n    def handle_starttag(self, tag, attrs):\n        attrs = dict(attrs)\n        if tag == "shreddit-post":\n            if self.depth:\n                self.depth += 1\n            elif self.post is None and attrs.get("id") == self.post_id:\n                self.post, self.depth = attrs, 1\n        if (\n            tag == "shreddit-player"\n            and self.depth == 1\n            and self.player is None\n            and attrs.get("post-id") == self.post_id\n            and attrs.get("src")\n        ):\n            self.player = attrs\n\n    def handle_endtag(self, tag):\n        if tag == "shreddit-post" and self.depth:\n            self.depth -= 1\n\n\ndef public_post_media(webpage, post_id):\n    parser = _PublicPost(post_id)\n    parser.feed(webpage)\n    if not parser.post or not parser.player:\n        return None\n    content = urlsplit(parser.post.get("content-href", ""))\n    source = urlsplit(parser.player["src"])\n    media_id = content.path.strip("/")\n    if (\n        content.scheme != "https"\n        or content.hostname != "v.redd.it"\n        or not media_id\n        or "/" in media_id\n        or source.scheme != "https"\n        or source.hostname != "v.redd.it"\n        or source.username\n        or source.password\n        or not source.path.startswith("/" + media_id + "/")\n    ):\n        return None\n    return parser.post, parser.player, media_id\n\n\nclass RedditPublicIE(RedditIE, plugin_name="vidfetch_public"):\n    def _real_extract(self, url):\n        try:\n            return super()._real_extract(url)\n        except ExtractorError as original:\n            detail = str(original).lower()\n            if any(\n                item in detail\n                for item in ("private subreddit", "quarantined", "removed", "deleted", "429")\n            ) or not any(\n                item in detail\n                for item in (\n                    "403",\n                    "unable to access the reddit api",\n                    "account authentication is required",\n                )\n            ):\n                raise\n            post_id = self._match_id(url)\n            public_url = "https://www.reddit.com" + urlsplit(url).path\n            webpage = self._download_webpage(\n                public_url,\n                post_id,\n                "Checking accessible Reddit post page",\n                fatal=False,\n                impersonate=True,\n            )\n            parsed = public_post_media(webpage or "", post_id)\n            if parsed is None:\n                raise original\n            post, player, media_id = parsed\n            source = player["src"]\n            if urlsplit(source).path.endswith(".m3u8"):\n                formats, subtitles = self._extract_m3u8_formats_and_subtitles(\n                    source, media_id, "mp4", m3u8_id="public-hls", fatal=False\n                )\n            elif urlsplit(source).path.endswith(".mpd"):\n                formats, subtitles = self._extract_mpd_formats_and_subtitles(\n                    source, media_id, mpd_id="public-dash", fatal=False\n                )\n            else:\n                raise original\n            if not formats:\n                raise original\n            duration = None\n            try:\n                duration = json.loads(player.get("packaged-media-json", "{}"))["playbackMp4s"][\n                    "duration"\n                ]\n            except (ValueError, TypeError, KeyError):\n                pass\n            return {\n                "id": media_id,\n                "display_id": post_id,\n                "title": post.get("post-title") or player.get("post-title") or post_id,\n                "uploader": post.get("author"),\n                "thumbnail": player.get("poster"),\n                "duration": duration,\n                "formats": formats,\n                "subtitles": subtitles,\n                "webpage_url": public_url,\n            }\n', 'vidfetch_bot/telegram.py': '"""Small asynchronous Telegram Bot API client; credentials never enter log output."""\n\nfrom __future__ import annotations\n\nimport asyncio\nimport mimetypes\nfrom pathlib import Path, PurePosixPath\nfrom urllib.parse import quote, urlsplit\n\nimport httpx\n\nCLOUD_UPLOAD_LIMIT = 49_000_000\nLOCAL_UPLOAD_LIMIT = 1_950_000_000  # Headroom below Telegram\'s documented 2000 MB ceiling.\n\n\nclass TelegramError(Exception):\n    def __init__(self, code: int, description: str, retry_after: int = 0):\n        self.code = code\n        self.retry_after = retry_after\n        # Telegram errors normally contain no token, but don\'t retain raw responses.\n        messages = {\n            401: "The bot token is invalid. Get a token from @BotFather and run again.",\n            409: "Another copy of this bot is running. Stop it before starting this copy.",\n            403: "Telegram could not deliver a message to this chat.",\n            429: "Telegram asked the bot to slow down. Try again shortly.",\n            503: "The local Telegram API server stopped. Restart the bot cell.",\n        }\n        self.description = description.lower()  # Used only to classify a video-format error.\n        super().__init__(messages.get(code, "Telegram could not complete the request. Try again."))\n\n\nclass Telegram:\n    def __init__(\n        self,\n        token: str,\n        *,\n        api_root: str = "https://api.telegram.org",\n        local_files: bool = False,\n        transport: httpx.AsyncBaseTransport | None = None,\n    ):\n        address = urlsplit(api_root)\n        if (\n            address.username\n            or address.password\n            or address.query\n            or address.fragment\n            or address.path not in {"", "/"}\n        ):\n            raise ValueError("Use a Telegram API origin without credentials or a path.")\n        loopback = address.hostname in {"127.0.0.1", "::1"}\n        if address.scheme != "https" and not (address.scheme == "http" and loopback):\n            raise ValueError("Telegram API connections require HTTPS or a loopback address.")\n        if local_files and not loopback:\n            raise ValueError("Local file delivery requires a server on the same machine.")\n        self.local_files = local_files\n        self.api_root = api_root.rstrip("/")\n        self.server_process = None\n        self.file_directory: Path | None = None\n        self._file_url = f"{api_root.rstrip(\'/\')}/file/bot{token}/"\n        self.upload_limit = LOCAL_UPLOAD_LIMIT if local_files else CLOUD_UPLOAD_LIMIT\n        self._client = httpx.AsyncClient(\n            base_url=f"{api_root.rstrip(\'/\')}/bot{token}/",\n            transport=transport,\n            timeout=httpx.Timeout(60, connect=20, write=600),\n            follow_redirects=False,\n            trust_env=not local_files,  # Loopback requests must not use an HTTP proxy.\n        )\n\n    async def close(self) -> None:\n        await self._client.aclose()\n\n    async def call(\n        self,\n        method: str,\n        data: dict | None = None,\n        *,\n        files: dict | None = None,\n        uploading: bool = False,\n    ):\n        if self.server_process is not None and self.server_process.returncode is not None:\n            raise TelegramError(503, "local server stopped")\n        options = {"timeout": httpx.Timeout(3600, connect=20, write=3600)} if uploading else {}\n        try:\n            if files:\n                response = await self._client.post(method, data=data, files=files, **options)\n            else:\n                response = await self._client.post(method, json=data or {}, **options)\n            body = response.json()\n        except (httpx.HTTPError, ValueError):\n            # httpx\'s exception text includes the URL, which includes the bot token.\n            raise TelegramError(0, "network") from None\n        if not body.get("ok"):\n            raise TelegramError(\n                int(body.get("error_code", response.status_code)),\n                str(body.get("description", "")),\n                int(body.get("parameters", {}).get("retry_after", 0)),\n            )\n        return body.get("result")\n\n    async def send_file(self, method: str, field: str, path: Path, payload: dict) -> None:\n        size = path.stat().st_size\n        if not 0 < size <= self.upload_limit:\n            raise TelegramError(413, "file size")\n        if self.local_files:\n            await self.call(method, {**payload, field: path.resolve().as_uri()}, uploading=True)\n        else:\n            content_type = mimetypes.guess_type(path.name)[0] or "application/octet-stream"\n            if path.suffix == ".m4a":\n                content_type = "audio/mp4"\n            with path.open("rb") as stream:\n                await self.call(\n                    method,\n                    payload,\n                    files={field: (path.name, stream, content_type)},\n                    uploading=True,\n                )\n\n    async def read_document(self, file_id: str, *, max_bytes: int) -> bytes:\n        """Bound incoming cookie data; never log download URLs or expose cached paths."""\n        if not isinstance(file_id, str) or not file_id or len(file_id) > 512 or max_bytes <= 0:\n            raise TelegramError(400, "invalid document")\n        info = await self.call("getFile", {"file_id": file_id})\n        if not isinstance(info, dict):\n            raise TelegramError(400, "invalid document")\n        size = info.get("file_size")\n        if size is not None and (type(size) is not int or not 0 < size <= max_bytes):\n            raise TelegramError(413, "document too large")\n        name = info.get("file_path")\n        if not isinstance(name, str) or not name:\n            raise TelegramError(400, "missing document")\n        if self.local_files:\n            source = None\n            try:\n                candidate = Path(name)\n                if not self.file_directory or not candidate.is_absolute() or candidate.is_symlink():\n                    raise TelegramError(400, "invalid local document")\n                candidate = candidate.resolve()\n                if not candidate.is_relative_to(self.file_directory.resolve()) or not candidate.is_file():\n                    raise TelegramError(400, "invalid local document")\n                relative = candidate.relative_to(self.file_directory.resolve())\n                if len(relative.parts) < 2 or relative.parts[0] == "temp":\n                    raise TelegramError(400, "not a downloaded document")\n                source = candidate\n                with source.open("rb") as stream:\n                    data = stream.read(max_bytes + 1)\n                if not 0 < len(data) <= max_bytes:\n                    raise TelegramError(413, "document too large")\n                return data\n            except OSError:\n                raise TelegramError(0, "document unavailable") from None\n            finally:\n                if source is not None:\n                    try:\n                        source.unlink(missing_ok=True)\n                    except OSError:\n                        pass\n        if PurePosixPath(name).is_absolute() or ".." in PurePosixPath(name).parts or "\\\\" in name:\n            raise TelegramError(400, "invalid document path")\n        try:\n            async with asyncio.timeout(45):\n                async with self._client.stream("GET", self._file_url + quote(name, safe="/")) as response:\n                    response.raise_for_status()\n                    # Redirects are not downloads and must not forward a credential-bearing request.\n                    if response.status_code != 200:\n                        raise TelegramError(400, "document unavailable")\n                    data = bytearray()\n                    async for chunk in response.aiter_bytes(64 * 1024):\n                        if len(data) + len(chunk) > max_bytes:\n                            raise TelegramError(413, "document too large")\n                        data.extend(chunk)\n                    if not data:\n                        raise TelegramError(400, "empty document")\n                    return bytes(data)\n        except (httpx.HTTPError, TimeoutError, ValueError):\n            raise TelegramError(0, "document unavailable") from None\n\n    async def upload_document(self, chat_id: int, path: Path, caption: str) -> None:\n        await self.send_file(\n            "sendDocument",\n            "document",\n            path,\n            {"chat_id": str(chat_id), "caption": caption, "parse_mode": "HTML"},\n        )\n\n    async def message(self, chat_id: int, text: str, *, keyboard: list | None = None) -> dict:\n        payload = {"chat_id": chat_id, "text": text, "link_preview_options": {"is_disabled": True}}\n        if keyboard is not None:\n            payload["reply_markup"] = {"inline_keyboard": keyboard}\n        return await self.call("sendMessage", payload)\n\n    async def edit(\n        self, chat_id: int, message_id: int, text: str, *, keyboard: list | None = None\n    ) -> None:\n        try:\n            await self.call(\n                "editMessageText",\n                {\n                    "chat_id": chat_id,\n                    "message_id": message_id,\n                    "text": text,\n                    "reply_markup": {"inline_keyboard": keyboard or []},\n                    "link_preview_options": {"is_disabled": True},\n                },\n            )\n        except TelegramError as error:\n            if error.code in {401, 409}:\n                raise\n            # Deleted status messages, duplicate edits and temporary network failures\n            # must not destroy a successfully downloaded/compressed file.\n\n    async def acknowledge(self, query_id: str, text: str = "") -> None:\n        try:\n            await self.call("answerCallbackQuery", {"callback_query_id": query_id, "text": text})\n        except TelegramError:\n            pass  # An old callback may expire before Telegram receives the answer.\n\n    async def upload(\n        self, chat_id: int, path: Path, caption: str, *, width: int, height: int, duration: int\n    ) -> None:\n        payload = {\n            "chat_id": str(chat_id),\n            "caption": caption,\n            "parse_mode": "HTML",\n            "supports_streaming": "true",\n            "width": str(width),\n            "height": str(height),\n            "duration": str(duration),\n        }\n        try:\n            await self.send_file("sendVideo", "video", path, payload)\n        except TelegramError as error:\n            # Only an explicit format rejection justifies resending as a document.\n            # Never retry ambiguous network/timeouts: the first upload may have arrived.\n            if error.code != 400 or not any(\n                word in error.description\n                for word in (\n                    "video_content_type",\n                    "video_content_invalid",\n                    "wrong type of the web page content",\n                )\n            ):\n                raise\n            await self.upload_document(chat_id, path, caption)\n\n    async def updates(self, offset: int):\n        return await self.call(\n            "getUpdates",\n            {"offset": offset, "timeout": 30, "allowed_updates": ["message", "callback_query"]},\n        )\n\n    async def upload_audio(\n        self, chat_id: int, path: Path, caption: str, *, title: str, duration: int\n    ) -> None:\n        payload = {\n            "chat_id": str(chat_id),\n            "caption": caption,\n            "parse_mode": "HTML",\n            "title": title,\n            "duration": str(duration),\n        }\n        await self.send_file("sendAudio", "audio", path, payload)\n\n    async def poll(self):\n        offset = 0\n        backoff = 1\n        while True:\n            try:\n                updates = await self.updates(offset)\n                backoff = 1\n                for update in updates:\n                    yield update\n                    offset = max(offset, update["update_id"] + 1)\n            except TelegramError as error:\n                if error.code in {401, 409, 503}:\n                    raise\n                await asyncio.sleep(min(max(backoff, error.retry_after), 60))\n                backoff = min(backoff * 2, 30)\n'}.items():
    destination = vidfetch_source_dir / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding="utf-8")
sys.path.insert(0, str(vidfetch_source_dir))
# Rerunning setup must load this notebook's version, not a cached older package.
for name in list(sys.modules):
    if name == "vidfetch_bot" or name.startswith("vidfetch_bot."):
        del sys.modules[name]
importlib.invalidate_caches()
from vidfetch_bot.local_api import install_server
from vidfetch_bot.cookies import CookieStore
if "vidfetch_cookie_dir" not in globals():
    vidfetch_cookie_dir = Path(tempfile.mkdtemp(prefix="vidfetch-cookies-"))
vidfetch_cookies = CookieStore(vidfetch_cookie_dir)
vidfetch_api_binary = install_server()
print("Setup complete. Configure optional site cookies, then start the bot.")


In [ ]:
#@title 2. Optional YouTube and Reddit cookies
# Stop the running bot before changing cookies. Cookie files stay in this temporary runtime.
youtube_cookies = "Keep" #@param ["Keep", "Import / replace", "Enable", "Disable", "Delete"]
reddit_cookies = "Keep" #@param ["Keep", "Import / replace", "Enable", "Disable", "Delete"]
from vidfetch_bot.colab_cookies import configure_cookies
configure_cookies(vidfetch_cookies, {
    "youtube.com": youtube_cookies,
    "reddit.com": reddit_cookies,
})


In [ ]:
#@title 3. Start the Telegram bot
# Optional: comma-separated numeric Telegram user IDs. Blank allows any private-chat user.
allowed_user_ids = "" #@param {type:"string"}

from vidfetch_bot.app import run_bot
from vidfetch_bot.credentials import read_secret
from vidfetch_bot.local_api import LocalAPIError, local_bot_api
from vidfetch_bot.telegram import TelegramError

async def start_vidfetch():
    try:
        allowed = {int(value.strip()) for value in allowed_user_ids.split(",") if value.strip()}
    except ValueError:
        print("Use numeric user IDs separated by commas, or leave the field blank.")
        return
    if vidfetch_cookies.has_enabled() and not allowed:
        print("Enter your Telegram user ID in allowed_user_ids before enabling cookies. To find it, disable cookies, start the bot and send /id.")
        return
    # Colab Secrets or hidden inputs keep credentials out of source and outputs.
    token = read_secret("TELEGRAM_BOT_TOKEN", "Bot token from @BotFather: ")
    api_id = read_secret("TELEGRAM_API_ID", "Telegram api_id from my.telegram.org: ")
    api_hash = read_secret("TELEGRAM_API_HASH", "Telegram api_hash from my.telegram.org: ")
    try:
        async with local_bot_api(
            token, api_id, api_hash, binary=vidfetch_api_binary
        ) as api:
            await run_bot(token, allowed_users=allowed, api=api, cookie_store=vidfetch_cookies)
    except (LocalAPIError, TelegramError) as error:
        print(str(error))

await start_vidfetch()
